# Week 5 Hands-On Lab (v2) — Open-Vocabulary Grounding: Boxes, Pixels, Words

**ESP3201 · formative hands-on lab.** Best on free-tier Colab with a **GPU (T4)
runtime**. The detector and segmenter also run on CPU (slower). The VLM section
offers a **free hosted API** path that needs no GPU at all, plus a local model
and an offline mock — so the whole notebook is runnable either way.

You will drive three models that all accept **open-vocabulary text** — words
nobody trained a fixed class list for — and watch how much your *phrasing*
changes what they report:

| Head | Model | Text in → out |
|---|---|---|
| **Detection** | OWL-ViT | text → **boxes** + scores |
| **Segmentation** | SAM 3 | text → **per-instance** masks + a presence score |
| **Segmentation (no text)** | SAM | **box** → per-pixel mask |
| **Language** | SmolVLM *or a hosted free-tier VLM* | text + image → **text** |

### The claim this lab tests

All three text-driven models are the same shape underneath: a **CLIP-style
text–image similarity engine** with a different head bolted on. If that is
true, they should share failure modes — no handling of negation, no way to
abstain, uncalibrated scores, shaky attribute binding, weak spatial relations —
and each head should add failures of its own.

Your job is to **find edge cases and say which kind each one is**: a failure of
the *head*, or a failure of the *shared backbone*. That distinction is what the
deliverable is graded on.

### Everything here is label-free

You will upload your own images, and nobody has annotated them. So every metric
in this lab compares **a model against itself under a changed prompt**. You do
not need to know what is in a photo to know that a model contradicted itself —
and self-contradiction is real evidence.

> **Report only numbers your own run produced.** The GPU is shared and models
> get updated; your figures may differ from anything quoted in class. That is
> fine and it is the point — quote yours.

## Setup

Run these two cells once. Both are collapsed — they are long and there is nothing to edit inside them.

In [ ]:
#@title Install + lab core (run me) { display-mode: "form" }
import os, sys, subprocess

# Pinned to a release verified against every model path in this notebook.
# If a future Colab image breaks this, loosen the pin and re-verify the four
# model loads before class -- do not silently drop the pin.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==5.14.1", "accelerate", "pillow", "matplotlib"],
               check=False)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cpu":
    print("NOTE: no GPU. OWL-ViT/SAM 3/SAM will run (slowly); for the VLM "
          "section use the hosted OpenRouter backend (0 GB), or switch the "
          "runtime to a T4 via Runtime > Change runtime type.")

# --- Week 5 v2 lab core, embedded directly (no repo clone) --------------------
# Canonical source: starter/grounding_lab_v2.py in the course repo, inlined by
# docs/_tools/build_week05_v2_nb.py. Cloning a support module from Colab is
# fragile: a session that already ran once before an update landed silently
# no-ops onto the stale cached copy instead of fetching the fix.

"""Week 5 lab core, v2 — open-vocabulary grounding across three model heads.

Canonical source for `notebooks/week05_open_vocab_grounding_colab.ipynb`. The
notebook embeds this file's contents directly in its setup cell (see the v1
lab's note: cloning a support module from Colab silently no-ops onto a stale
cached copy if the session ran once before an update landed). Keep the two in
sync by regenerating the notebook with `docs/_tools/build_week05_v2_nb.py`
rather than hand-editing the notebook's setup cell.

Three heads on one backbone (all three text encoders are CLIP-family):
  * OWL-ViT   — text prompt -> boxes                    (`OwlDetector`)
  * SAM 3     — text prompt -> instances + presence     (`Sam3Segmenter`)
  * SAM       — box prompt  -> pixels, reads no text    (`SamSegmenter`)
  * VLM       — text+image  -> text                     (`HFVLM` / `OpenRouterVLM`
                                                         / `GeminiVLM`)

Every metric here is LABEL-FREE: it compares a model against itself under a
changed prompt, so it works on any image a student uploads with no annotation.
"""

from __future__ import annotations

import hashlib
import itertools
import json
import os
import re
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

class _Freeable:
    """Gives every model wrapper a readable error after `free_model()`."""

    def __getattr__(self, name):
        # Only reached when normal attribute lookup already failed.
        if name in ("model", "processor") and self.__dict__.get("_freed"):
            raise RuntimeError(
                f"{type(self).__name__} was released by free_model() to reclaim "
                f"VRAM. Re-create it (e.g. {type(self).__name__}(device=DEVICE)) "
                "before using it again."
            )
        raise AttributeError(f"{type(self).__name__!r} object has no attribute {name!r}")


# --------------------------------------------------------------------------- #
# Geometry / label-free comparison primitives
# --------------------------------------------------------------------------- #


def clip_box(box: Sequence[float], width: int, height: int) -> Tuple[float, ...]:
    """Clamp an (x0, y0, x1, y1) box to the image. OWL-ViT returns boxes that
    run slightly outside the frame; drawing those raises or looks broken."""
    x0, y0, x1, y1 = box
    return (max(0.0, min(float(x0), width)), max(0.0, min(float(y0), height)),
            max(0.0, min(float(x1), width)), max(0.0, min(float(y1), height)))


def iou(a: Optional[Sequence[float]], b: Optional[Sequence[float]]) -> float:
    """Intersection-over-union of two (x0, y0, x1, y1) boxes.

    Returns 0.0 if either box is missing -- "the model found nothing under this
    phrasing" is a real answer, not an error, and scores as total disagreement.
    """
    if a is None or b is None:
        return 0.0
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    iw, ih = max(0.0, ix1 - ix0), max(0.0, iy1 - iy0)
    inter = iw * ih
    area_a = max(0.0, ax1 - ax0) * max(0.0, ay1 - ay0)
    area_b = max(0.0, bx1 - bx0) * max(0.0, by1 - by0)
    union = area_a + area_b - inter
    return round(inter / union, 3) if union > 0 else 0.0


def mask_iou(m1: np.ndarray, m2: np.ndarray) -> float:
    """IoU of two boolean masks of the same shape."""
    m1, m2 = np.asarray(m1).astype(bool), np.asarray(m2).astype(bool)
    union = np.logical_or(m1, m2).sum()
    return round(float(np.logical_and(m1, m2).sum() / union), 3) if union else 0.0


def containment(part: np.ndarray, whole: np.ndarray) -> float:
    """Fraction of `part` that lies inside `whole`.

    A model that understands part-of should put ~all of "a wheel" inside "a
    car". Containment near 1 with a small area ratio is correct part/whole
    behaviour; containment near 1 with an area ratio near 1 means the model
    just segmented the whole object again and ignored the part word.
    """
    part, whole = np.asarray(part).astype(bool), np.asarray(whole).astype(bool)
    return round(float(np.logical_and(part, whole).sum() / part.sum()), 3) if part.sum() else 0.0


# --------------------------------------------------------------------------- #
# Detection head — OWL-ViT
# --------------------------------------------------------------------------- #


@dataclass
class Det:
    """One detection. `prompt` is the text query that produced it."""
    prompt: str
    score: float
    box: Tuple[float, float, float, float]


class OwlDetector(_Freeable):
    """Open-vocabulary detector. Text queries in, boxes + scores out.

    Pinned to `google/owlvit-base-patch32` (see pinned_models.md). `OWLv2`
    (`google/owlv2-base-patch16-ensemble`) is a drop-in swap and is worth trying
    -- it is the stronger model the lecture also cites.
    """

    def __init__(self, model_id: str = "google/owlvit-base-patch32",
                 device: str = "cuda"):
        import torch
        from transformers import OwlViTForObjectDetection, OwlViTProcessor
        self.torch = torch
        self.model_id = model_id
        self.processor = OwlViTProcessor.from_pretrained(model_id)
        self.model = OwlViTForObjectDetection.from_pretrained(model_id).to(device).eval()
        self.device = device

    def _raw(self, image, prompts: Sequence[str], threshold: float):
        inputs = self.processor(text=[list(prompts)], images=image,
                                return_tensors="pt").to(self.device)
        with self.torch.no_grad():
            outputs = self.model(**inputs)
        sizes = self.torch.tensor([[image.height, image.width]]).to(self.device)
        # transformers renamed this between 4.x and 5.x; accept either.
        fn = (getattr(self.processor, "post_process_grounded_object_detection", None)
              or getattr(self.processor, "post_process_object_detection"))
        return fn(outputs=outputs, threshold=threshold, target_sizes=sizes)[0]

    def detect(self, image, prompts: Sequence[str],
               threshold: float = 0.1) -> List[Det]:
        """Boxes above `threshold`, highest score first."""
        prompts = list(prompts)
        res = self._raw(image, prompts, threshold)
        dets = [
            Det(prompt=prompts[int(label)], score=round(float(score), 3),
                box=clip_box(box.tolist(), image.width, image.height))
            for box, score, label in zip(res["boxes"], res["scores"], res["labels"])
        ]
        return sorted(dets, key=lambda d: -d.score)

    def max_scores(self, image, prompts: Sequence[str]) -> Dict[str, float]:
        """Best score per prompt with NO threshold applied.

        Use this for absent objects: thresholding hides the fact that the model
        still ranked *something* as the best "a giraffe" in a kitchen. The
        detector never abstains -- it only ever scores low.

        Each prompt gets its OWN forward pass. That is deliberate and it is not
        just caution: OWL-ViT scores all 576 candidate boxes against every
        query, then reports each box under its single best-matching query only
        (see `vocabulary_competition`). Asked as one batch, a prompt that loses
        every box to a competing word would read as 0.0 here -- which is a fact
        about your word list, not about the image. One pass per prompt gives the
        prompt's own score, uncontaminated.
        """
        best = {}
        for p in prompts:
            res = self._raw(image, [p], 0.0)
            scores = [float(s) for s in res["scores"]]
            best[p] = round(max(scores), 3) if scores else 0.0
        return best

    def vocabulary_competition(self, image, target: str, competitor: str,
                               threshold: float = 0.02) -> Dict[str, object]:
        """What adding one word to your vocabulary does to another word's boxes.

        OWL-ViT assigns each candidate box to its argmax query, so the labels
        you get back depend on the whole prompt list, not just on the image.
        Query "an orange" alone and a region is an orange; add "a lemon" to the
        list and the same region is reported as a lemon instead. Nothing about
        the image or either score changed -- only the competition did.

        `boxes_lost_by_target` is the count of boxes the target reported alone
        that the competitor takes over once both are queried together.
        """
        alone_t = self.detect(image, [target], threshold=threshold)
        alone_c = self.detect(image, [competitor], threshold=threshold)
        together = self.detect(image, [target, competitor], threshold=threshold)
        t_boxes_alone = {tuple(round(v) for v in d.box) for d in alone_t}
        t_boxes_together = {tuple(round(v) for v in d.box)
                            for d in together if d.prompt == target}
        return {
            "target": target, "competitor": competitor,
            "score_target_alone": alone_t[0].score if alone_t else 0.0,
            "score_competitor_alone": alone_c[0].score if alone_c else 0.0,
            "n_target_alone": len(alone_t),
            "n_target_together": len(t_boxes_together),
            "boxes_lost_by_target": len(t_boxes_alone - t_boxes_together),
            "dets_alone": alone_t, "dets_together": together,
        }

    def top_box(self, image, prompt: str, threshold: float = 0.0
                ) -> Tuple[Optional[Tuple[float, ...]], float]:
        """Highest-scoring box for one prompt, or (None, 0.0)."""
        dets = [d for d in self.detect(image, [prompt], threshold=threshold)]
        return (dets[0].box, dets[0].score) if dets else (None, 0.0)


# --------------------------------------------------------------------------- #
# Segmentation heads — SAM 3 (text -> instances) and SAM (box -> pixels)
# --------------------------------------------------------------------------- #


@dataclass
class Instance:
    """One segmented instance of a concept."""
    concept: str
    score: float
    box: Tuple[float, float, float, float]
    mask: np.ndarray            # boolean (H, W)


class Sam3Segmenter(_Freeable):
    """Text-prompted **instance** segmentation, with an explicit presence head.

    One concept per query. Returns a separate mask, box and score for each
    instance of that concept, plus a single `presence` score for "is this
    concept in the image at all" -- a capability neither OWL-ViT nor CLIPSeg
    has. Both are worth measuring, and they can disagree.

    WHICH CHECKPOINT THIS IS, AND WHY IT IS NOT THE LECTURE'S MODEL
    ---------------------------------------------------------------
    Default is `vil-uob/sam3-litetext-s0`: SAM 3 with its heavy text encoder
    distilled down to a MobileCLIP-based one (~88 % fewer text-encoder
    parameters), released by the Visual Information Lab as a **third party**,
    not by Meta.

    Two things to state in any report built on this:

    1. **It is not SegCLIP.** The lecture teaches SegCLIP (Luo et al., arXiv
       2211.14813) -- annotation-free open-vocabulary *semantic* segmentation.
       Its checkpoint needs torch 1.8 / mmcv-full 1.3.14 / Python 3.8 and does
       not install on current Colab. This model is a different architecture
       class (a DETR-style instance decoder, mask-supervised at scale), so
       nothing measured here speaks to SegCLIP's annotation-free claim.
    2. **The licence is unsettled.** The card says Apache-2.0, but the weights
       derive from SAM 3, whose official release (`facebook/sam3`) is gated
       under Meta's own SAM 3 licence. Treat the Apache-2.0 label as possibly
       mistaken. This lab downloads at runtime and redistributes nothing.

    What makes it a good subject anyway: its text encoder is still
    **CLIP-family**. So when negation and superordinate categories fail here
    exactly as they failed in OWL-ViT, that is not a coincidence to note -- it
    is the shared text encoder, and you can say so mechanistically.
    """

    def __init__(self, model_id: str = "vil-uob/sam3-litetext-s0",
                 device: str = "cuda"):
        import torch
        from transformers import AutoModel, AutoProcessor
        self.torch = torch
        self.model_id = model_id
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.model = AutoModel.from_pretrained(model_id).to(device).eval()
        self.device = device

    def _forward(self, image, concept: str):
        # One concept per forward pass: this model does NOT accept several
        # concepts in one string. Verified -- "keyboard. laptop." returns zero
        # instances, silently, rather than erroring.
        inputs = self.processor(images=image, text=concept,
                                return_tensors="pt").to(self.device)
        with self.torch.no_grad():
            return self.processor, self.model(**inputs), inputs

    def segment(self, image, concept: str, threshold: float = 0.5) -> List[Instance]:
        """Instances of `concept` scoring above `threshold`, best first."""
        proc, out, _ = self._forward(image, concept)
        res = proc.post_process_instance_segmentation(
            out, threshold=threshold,
            target_sizes=[(image.height, image.width)])[0]
        insts = [
            Instance(concept=concept, score=round(float(s), 3),
                     box=clip_box(b.tolist(), image.width, image.height),
                     mask=m.detach().cpu().numpy().astype(bool))
            for s, b, m in zip(res["scores"], res["boxes"], res["masks"])
        ]
        return sorted(insts, key=lambda i: -i.score)

    def presence(self, image, concept: str) -> float:
        """P(concept appears in this image), from the model's presence head.

        This is the abstention mechanism the older models lack: a single number
        for "is it here", separate from where it is. A well-behaved model puts
        far-fetched concepts near 0 -- and the interesting failures are the
        concepts that land in the middle.
        """
        _, out, _ = self._forward(image, concept)
        return round(float(self.torch.sigmoid(out.presence_logits).flatten()[0]), 4)

    def union_mask(self, image, concept: str, threshold: float = 0.5) -> np.ndarray:
        """All instances of `concept` merged into one boolean mask.

        Use this when you want to compare against a *semantic* mask (every
        pixel of this concept) rather than reason per instance.
        """
        insts = self.segment(image, concept, threshold)
        out = np.zeros((image.height, image.width), bool)
        for i in insts:
            out |= i.mask
        return out

    def masks(self, image, concepts: Sequence[str],
              threshold: float = 0.5) -> Dict[str, np.ndarray]:
        """concept -> union boolean mask, for the mask-comparison probes."""
        return {c: self.union_mask(image, c, threshold) for c in concepts}


class SamSegmenter(_Freeable):
    """Promptable segmentation from a BOX. Reads no text at all.

    Pairing this with OWL-ViT gives the 'detect then segment' pipeline real
    robot stacks use. The comparison worth making: the text prompt reaches SAM
    only through the box the detector chose, so every language failure in the
    detector is inherited by the mask -- while SAM's own boundaries can be
    excellent on an object the detector named wrongly.
    """

    def __init__(self, model_id: str = "facebook/sam-vit-base",
                 device: str = "cuda"):
        import torch
        from transformers import SamModel, SamProcessor
        self.torch = torch
        self.model_id = model_id
        self.processor = SamProcessor.from_pretrained(model_id)
        self.model = SamModel.from_pretrained(model_id).to(device).eval()
        self.device = device

    def mask_from_box(self, image, box: Sequence[float]) -> np.ndarray:
        """Boolean mask (H, W) for the object inside `box`."""
        inputs = self.processor(image, input_boxes=[[list(map(float, box))]],
                                return_tensors="pt").to(self.device)
        with self.torch.no_grad():
            outputs = self.model(**inputs, multimask_output=False)
        masks = self.processor.image_processor.post_process_masks(
            outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(),
            inputs["reshaped_input_sizes"].cpu())
        return masks[0][0][0].numpy().astype(bool)


# --------------------------------------------------------------------------- #
# Language head — VLM backends
# --------------------------------------------------------------------------- #


_YES_RE = re.compile(r"\b(yes|yeah|correct)\b")
_NO_RE = re.compile(r"\b(no|not|none|isn't|aren't|doesn't|don't)\b")


def parse_yes_no(text: str) -> Optional[str]:
    """'yes' / 'no' / None when the model said neither.

    None matters: a model that answers "I cannot tell" is behaving BETTER than
    one that guesses, and collapsing that to a coin flip would hide it.

    Matching is on WORD BOUNDARIES, which is not fussiness -- a plain substring
    test reads "no" inside "cannot", "nothing" and "know", so "I cannot tell"
    would be scored as a confident "no". That single bug would inflate every
    contradiction and capitulation rate in the lab.
    """
    t = (text or "").strip().lower()
    head = t[:60]
    has_yes, has_no = bool(_YES_RE.search(head)), bool(_NO_RE.search(head))
    if has_yes and not has_no:
        return "yes"
    if has_no and not has_yes:
        return "no"
    # Both or neither present: fall back to how the answer OPENS, which is what
    # an instruction-following model puts its verdict in.
    if t.startswith("yes"):
        return "yes"
    if t.startswith("no"):
        return "no"
    return None


class HFVLM(_Freeable):
    """Small local VLM through the transformers image-text-to-text interface.

    Verified on `HuggingFaceTB/SmolVLM-Instruct` (~4.9 GB peak in fp16; see
    pinned_models.md). The same path works for other instruct VLMs behind
    `AutoModelForImageTextToText`; re-measure memory if you swap checkpoints.
    """

    def __init__(self, model_id: str = "HuggingFaceTB/SmolVLM-Instruct",
                 device: str = "cuda", dtype: str = "float16",
                 max_new_tokens: int = 24):
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor
        self.torch = torch
        self.model_id = model_id
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.model = AutoModelForImageTextToText.from_pretrained(
            model_id, dtype=getattr(torch, dtype)).to(device).eval()
        self.device = device
        self.max_new_tokens = max_new_tokens

    def ask(self, image, question: str) -> str:
        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": question}]}]
        prompt = self.processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = self.processor(text=prompt, images=[image], return_tensors="pt").to(self.device)
        n_in = inputs["input_ids"].shape[1]
        with self.torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens,
                                      do_sample=False)
        return self.processor.batch_decode(out[:, n_in:], skip_special_tokens=True)[0].strip()


def get_openrouter_key() -> str:
    """Resolve OPENROUTER_API_KEY: Colab Secrets -> environment -> prompt.

    Same resolution order as the Week 9 agent notebook, deliberately -- a
    student who set the secret up for that lab does not have to do it again.
    """
    try:
        from google.colab import userdata
        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            print("Using OPENROUTER_API_KEY from Colab Secrets.")
            return key
    except Exception:  # noqa: BLE001
        pass
    key = os.environ.get("OPENROUTER_API_KEY", "")
    if key:
        print("Using OPENROUTER_API_KEY from the environment.")
        return key
    from getpass import getpass
    print("No OPENROUTER_API_KEY found in Colab Secrets or the environment.")
    key = getpass("Paste your OpenRouter API key (input hidden, not saved to disk): ").strip()
    if not key:
        raise RuntimeError(
            "An OpenRouter API key is required. Free keys: "
            "https://openrouter.ai/settings/keys")
    return key


class OpenRouterVLM:
    """Hosted VLM over OpenRouter's OpenAI-compatible endpoint. No GPU needed.

    This is the zero-VRAM path: pick it and the whole VLM section runs over
    HTTP, leaving the entire T4 for the vision models. It is also the only
    backend where swapping in a *different* VLM is a one-line change, so it is
    the cheapest way to answer "does my finding survive a different model?".

    The image is sent inline as a base64 data URL in the standard OpenAI vision
    content format, so the model must accept image input -- most free text
    models do not. `list_free_vision_models()` prints what currently advertises
    it.

    Replies are cached on (model, image, question). That is not an optimisation:
    re-running a cell is the normal way to work in a notebook, and without a
    cache every re-run re-spends a metered free-tier quota.

    PIN THIS: fix the exact model id per offering and confirm it answers before
    class. Free-tier availability changes without notice, and a model that
    silently returns text-only refusals will quietly corrupt every metric.
    """

    ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"
    #: measured: 6 concurrent requests, 18 questions in 12.4 s, no 429s.
    max_workers = 6

    def __init__(self, model_id: str = "", api_key: Optional[str] = None,
                 temperature: float = 0.0, max_tokens: int = 32,
                 timeout: int = 120, jpeg_quality: int = 90,
                 max_retries: int = 4, retry_base_delay: float = 2.0,
                 sleep=None):
        import threading
        import time as _time
        if not model_id:
            raise ValueError(
                "Set model_id to a pinned OpenRouter model that accepts images "
                "(see list_free_vision_models()).")
        self.model_id = model_id
        self.api_key = (api_key or os.environ.get("OPENROUTER_API_KEY", "")
                        or get_openrouter_key())
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.timeout = timeout
        self.jpeg_quality = jpeg_quality
        self.max_retries = max_retries
        self.retry_base_delay = retry_base_delay
        self._sleep = sleep or _time.sleep     # injectable so tests do not wait
        self._cache: Dict[str, str] = {}
        self._lock = threading.Lock()
        self.n_calls = 0
        self.n_cache_hits = 0
        self.n_retries = 0
        self.served: Dict[str, int] = {}

    def _data_url(self, image) -> str:
        import base64
        import io
        buf = io.BytesIO()
        image.convert("RGB").save(buf, format="JPEG", quality=self.jpeg_quality)
        return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode("ascii")

    @staticmethod
    def _image_key(image) -> str:
        import hashlib
        import io
        buf = io.BytesIO()
        image.convert("RGB").save(buf, format="PNG")
        return hashlib.sha256(buf.getvalue()).hexdigest()[:16]

    def ask(self, image, question: str) -> str:
        key = f"{self.model_id}|{self._image_key(image)}|{question}"
        with self._lock:
            hit = self._cache.get(key)
            if hit is not None:
                self.n_cache_hits += 1
                return hit
        text, served = self._request(image, question)
        with self._lock:
            self._cache[key] = text
            self.n_calls += 1
            self.served[served or self.model_id] = self.served.get(served or self.model_id, 0) + 1
        return text

    def _request(self, image, question: str) -> Tuple[str, Optional[str]]:
        import requests
        payload = {
            "model": self.model_id,
            "messages": [{"role": "user", "content": [
                {"type": "text", "text": question},
                {"type": "image_url", "image_url": {"url": self._data_url(image)}},
            ]}],
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
        }
        headers = {"Authorization": f"Bearer {self.api_key}",
                   "Content-Type": "application/json"}
        last = None
        for attempt in range(self.max_retries):
            r = requests.post(self.ENDPOINT, headers=headers, json=payload,
                              timeout=self.timeout)
            # 429 = rate limited, 5xx = transient. A whole class hitting a free
            # tier at once WILL see 429s; failing over one would throw away
            # every answer collected so far.
            if r.status_code == 429 or r.status_code >= 500:
                last = f"HTTP {r.status_code}"
                self.n_retries += 1
                self._sleep(self.retry_base_delay * (2 ** attempt))
                continue
            r.raise_for_status()
            data = r.json()
            if "choices" not in data:
                # OpenRouter reports quota/routing problems as HTTP 200 with an
                # error body. Returning "" would be logged as a refusal.
                raise RuntimeError(f"OpenRouter returned no choices: {str(data)[:300]}")
            # `data["model"]` is what actually served the request, which can
            # differ from what was asked for (aliases, routers, fallbacks).
            return ((data["choices"][0]["message"].get("content") or "").strip(),
                    data.get("model"))
        raise RuntimeError(
            f"OpenRouter still failing after {self.max_retries} attempts ({last}). "
            "Free-tier rate limit or outage -- wait and re-run; cached answers are kept.")

    def served_report(self) -> None:
        """Print which checkpoint(s) actually answered.

        Ask a *router* and several models can answer inside one run, making the
        metrics a blend that no single checkpoint can be credited with. Check
        this before quoting any result.
        """
        total = sum(self.served.values()) or 1
        print(f"requested: {self.model_id}   calls={self.n_calls} "
              f"cache_hits={self.n_cache_hits} retries={self.n_retries}")
        for m, n in sorted(self.served.items(), key=lambda kv: -kv[1]):
            print(f"  served by {m:<50} {n:>4} ({n / total:.0%})")
        if len(self.served) > 1:
            print("  WARNING: more than one model answered. These metrics are a "
                  "blend and cannot be attributed to one checkpoint.")


def list_free_vision_models(limit: int = 20) -> List[str]:
    """Print OpenRouter models that are free AND advertise image input.

    Queried live, because this list changes: pin whatever you verify today and
    re-check before each offering rather than trusting a hard-coded id.
    """
    import requests
    r = requests.get("https://openrouter.ai/api/v1/models", timeout=60)
    r.raise_for_status()
    out = []
    for m in r.json().get("data", []):
        mid = m.get("id", "")
        mods = (m.get("architecture") or {}).get("input_modalities") or []
        pricing = m.get("pricing") or {}
        is_free = mid.endswith(":free") or (
            str(pricing.get("prompt", "1")) in ("0", "0.0") and
            str(pricing.get("completion", "1")) in ("0", "0.0"))
        if is_free and "image" in mods:
            out.append(mid)
            if len(out) <= limit:
                print(f"  {mid:<52} ctx={m.get('context_length')} in={mods}")
    print(f"\n{len(out)} free model(s) currently advertise image input.")
    return out


class GeminiVLM:
    """Free-tier hosted VLM. PIN THIS: confirm the exact model id and that your
    key works before class -- ids and free-tier terms change."""

    def __init__(self, model_id: str = "", api_key: Optional[str] = None):
        import google.generativeai as genai
        if not model_id:
            raise ValueError("Set model_id to a pinned Gemini model (see the notebook).")
        genai.configure(api_key=api_key or os.environ["GOOGLE_API_KEY"])
        self.model = genai.GenerativeModel(model_id)
        self.model_id = model_id

    def ask(self, image, question: str) -> str:
        return (self.model.generate_content([question, image]).text or "").strip()


# --------------------------------------------------------------------------- #
# Label-free consistency probes — DETECTOR
# --------------------------------------------------------------------------- #

QUESTION_TEMPLATES = [
    "Is there a {obj} in the image? Answer yes or no.",
    "Can you see a {obj}? Answer yes or no.",
    "Does this image contain a {obj}? Answer yes or no.",
    "Is a {obj} present in this picture? Answer yes or no.",
]

PROMPT_TEMPLATES = ["{obj}", "a {obj}", "a photo of a {obj}", "an image of a {obj}"]


def synonym_stability(detector: OwlDetector, image, phrase_a: str, phrase_b: str,
                      threshold: float = 0.1) -> Dict[str, object]:
    """Two phrases a human would call interchangeable ("couch"/"sofa").

    A grounded detector should put the same box in the same place with a
    similar score. `top_box_iou` near 1 and `score_delta` near 0 = stable;
    a large `score_delta` with a high IoU is the interesting case -- it found
    the same thing but is far less confident it has the right *word* for it.
    """
    box_a, score_a = detector.top_box(image, phrase_a, threshold)
    box_b, score_b = detector.top_box(image, phrase_b, threshold)
    return {
        "phrase_a": phrase_a, "phrase_b": phrase_b,
        "score_a": score_a, "score_b": score_b,
        "score_delta": round(abs(score_a - score_b), 3),
        "top_box_iou": iou(box_a, box_b),
        "n_boxes_a": len(detector.detect(image, [phrase_a], threshold)),
        "n_boxes_b": len(detector.detect(image, [phrase_b], threshold)),
    }


def phrase_set_consistency(detector: OwlDetector, image, phrases: Sequence[str],
                           threshold: float = 0.1) -> Dict[str, object]:
    """Stability of one object's box across several phrasings of the same query.

    `mean_pairwise_iou` is the headline number: 1.0 means wording did not move
    the box at all, 0.0 means every phrasing found something different (or
    nothing). Needs no ground truth -- it is the model against itself.
    """
    phrases = list(phrases)
    tops = {p: detector.top_box(image, p, threshold) for p in phrases}
    ious = [iou(tops[a][0], tops[b][0]) for a, b in itertools.combinations(phrases, 2)]
    scores = [tops[p][1] for p in phrases]
    return {
        "phrases": phrases,
        "scores": {p: tops[p][1] for p in phrases},
        "mean_pairwise_iou": round(float(np.mean(ious)), 3) if ious else 1.0,
        "min_pairwise_iou": round(float(np.min(ious)), 3) if ious else 1.0,
        "score_spread": round(float(max(scores) - min(scores)), 3) if scores else 0.0,
        "n_found": sum(1 for p in phrases if tops[p][0] is not None),
    }


def negation_probe(detector: OwlDetector, image, obj: str,
                   threshold: float = 0.1) -> Dict[str, object]:
    """"a {obj}" vs "not a {obj}" vs "no {obj}".

    CLIP-style text encoders have no operator for negation. If the boxes are
    identical (`iou_affirm_vs_negated` near 1), the word "not" did nothing --
    the model matched on the noun and ignored the logic. That is a *backbone*
    failure, and you should expect to see it in the segmenter too.
    """
    affirm = f"a {obj}"
    results = {}
    for name, phrase in (("affirmed", affirm), ("negated", f"not a {obj}"),
                         ("negated_alt", f"no {obj}")):
        box, score = detector.top_box(image, phrase, threshold)
        results[name] = {"phrase": phrase, "score": score, "box": box}
    return {
        "obj": obj,
        "detail": results,
        "iou_affirm_vs_negated": iou(results["affirmed"]["box"], results["negated"]["box"]),
        "iou_affirm_vs_negated_alt": iou(results["affirmed"]["box"],
                                         results["negated_alt"]["box"]),
        "score_drop_when_negated": round(
            results["affirmed"]["score"] - results["negated"]["score"], 3),
    }


def threshold_sweep(detector: OwlDetector, image, prompts: Sequence[str],
                    thresholds: Sequence[float] = (0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5)
                    ) -> List[Dict[str, object]]:
    """How many boxes survive at each score cut. The 'number of objects' this
    model reports is a threshold you chose, not something it measured."""
    return [{"threshold": t, "n_boxes": len(detector.detect(image, prompts, threshold=t))}
            for t in thresholds]


def absent_object_probe(detector: OwlDetector, image, absent: Sequence[str],
                        present: Sequence[str] = ()) -> Dict[str, object]:
    """Best score for things that are NOT in the image, alongside things that are.

    The detector's version of hallucination. It cannot say "nothing here" --
    it always returns a ranked box. What you are checking is whether the
    absent-object scores are *separable* from the present-object ones, because
    if they overlap, no single threshold can tell them apart.
    """
    absent, present = list(absent), list(present)
    scores = detector.max_scores(image, absent + present) if (absent or present) else {}
    a_vals = [scores[p] for p in absent] or [0.0]
    p_vals = [scores[p] for p in present] or [0.0]
    return {
        "absent_scores": {p: scores[p] for p in absent},
        "present_scores": {p: scores[p] for p in present},
        "max_absent_score": round(max(a_vals), 3),
        "min_present_score": round(min(p_vals), 3),
        # >0 means a threshold exists that separates them on THIS image.
        "separation_margin": round(min(p_vals) - max(a_vals), 3) if present and absent else None,
    }


# --------------------------------------------------------------------------- #
# Label-free consistency probes — SEGMENTER
# --------------------------------------------------------------------------- #


def presence_probe(segmenter: Sam3Segmenter, image, present: Sequence[str],
                   absent: Sequence[str]) -> Dict[str, object]:
    """The presence head against concepts you have labelled present / absent.

    This is the one probe in the lab that is NOT label-free -- you supply the
    two lists by looking at the image. It earns its place because it is the only
    way to ask the question that matters: is the model's own "is it here" score
    *separable* between things that are and are not there?

    `separation_margin` = lowest present score minus highest absent score.
    Negative means the two sets overlap, so no threshold can split them.
    """
    present, absent = list(present), list(absent)
    p_scores = {c: segmenter.presence(image, c) for c in present}
    a_scores = {c: segmenter.presence(image, c) for c in absent}
    p_vals = list(p_scores.values()) or [0.0]
    a_vals = list(a_scores.values()) or [0.0]
    return {
        "present_presence": p_scores, "absent_presence": a_scores,
        "min_present": round(min(p_vals), 4), "max_absent": round(max(a_vals), 4),
        "separation_margin": (round(min(p_vals) - max(a_vals), 4)
                              if present and absent else None),
    }


def presence_vs_instances(segmenter: Sam3Segmenter, image, concept: str,
                          threshold: float = 0.5) -> Dict[str, object]:
    """Does the presence head agree with the instance head?

    Two outputs of the SAME model answering versions of one question. A high
    presence score with zero instances above threshold means the model believes
    the concept is there but will not commit to where -- so a pipeline that only
    reads masks silently discards what the model actually thinks. Whether you
    call that a bug depends on which output your robot consumes.
    """
    p = segmenter.presence(image, concept)
    insts = segmenter.segment(image, concept, threshold)
    return {
        "concept": concept, "presence": p, "n_instances": len(insts),
        "top_instance_score": insts[0].score if insts else 0.0,
        "disagrees": bool(p > 0.5 and not insts),
    }


def instance_count_consistency(segmenter: Sam3Segmenter, image,
                               phrases: Sequence[str], threshold: float = 0.5
                               ) -> Dict[str, object]:
    """Same object, several phrasings -- as instance counts and union masks.

    Two ways this can wobble that a box-based probe cannot see: the model can
    keep the same pixels but split them into a different NUMBER of instances,
    or keep the count and move the pixels. `count_spread` and
    `mean_pairwise_iou` separate those.
    """
    phrases = list(phrases)
    per = {p: segmenter.segment(image, p, threshold) for p in phrases}
    masks = {}
    for p, insts in per.items():
        m = np.zeros((image.height, image.width), bool)
        for i in insts:
            m |= i.mask
        masks[p] = m
    counts = {p: len(per[p]) for p in phrases}
    areas = {p: round(float(masks[p].mean()), 4) for p in phrases}
    pairs = {f"{a} | {b}": mask_iou(masks[a], masks[b])
             for a, b in itertools.combinations(phrases, 2)}
    vals = list(pairs.values())
    return {
        "phrases": phrases, "n_instances": counts, "area_fraction": areas,
        "presence": {p: segmenter.presence(image, p) for p in phrases},
        "pairwise_mask_iou": pairs,
        "mean_pairwise_iou": round(float(np.mean(vals)), 3) if vals else 1.0,
        "count_spread": max(counts.values()) - min(counts.values()) if counts else 0,
        "area_spread": round(max(areas.values()) - min(areas.values()), 4) if areas else 0.0,
    }


def score_threshold_sweep(segmenter: Sam3Segmenter, image, concept: str,
                          thresholds: Sequence[float] = (0.05, 0.1, 0.3, 0.5, 0.7)
                          ) -> List[Dict[str, float]]:
    """Instances and total area as the score cut moves.

    Run this on a concept the model appears to abstain on. If instances appear
    as you lower the cut, the model had candidates all along and the THRESHOLD
    did the abstaining -- which is a very different safety story from a model
    that genuinely has nothing to offer.
    """
    rows = []
    for t in thresholds:
        insts = segmenter.segment(image, concept, float(t))
        total = np.zeros((image.height, image.width), bool)
        for i in insts:
            total |= i.mask
        rows.append({"threshold": float(t), "n_instances": len(insts),
                     "total_area": round(float(total.mean()), 4),
                     "max_score": insts[0].score if insts else 0.0})
    return rows


def part_whole_probe(segmenter: Sam3Segmenter, image, part: str, whole: str,
                     threshold: float = 0.5) -> Dict[str, object]:
    """Does the model distinguish a part from the object containing it?

    Correct behaviour: high containment, small area ratio. Two ways to fail:
    return the whole object again (containment high AND area ratio near 1), or
    return nothing for the part while still scoring it present -- which is why
    both presence scores are reported here too.
    """
    masks = segmenter.masks(image, [part, whole], threshold=threshold)
    a_part = float(masks[part].mean())
    a_whole = float(masks[whole].mean())
    return {
        "part": part, "whole": whole,
        "part_area_fraction": round(a_part, 4), "whole_area_fraction": round(a_whole, 4),
        "area_ratio_part_over_whole": round(a_part / a_whole, 3) if a_whole else None,
        "containment_part_in_whole": containment(masks[part], masks[whole]),
        "mask_iou": mask_iou(masks[part], masks[whole]),
        "presence_part": segmenter.presence(image, part),
        "presence_whole": segmenter.presence(image, whole),
    }


def negation_probe_seg(segmenter: Sam3Segmenter, image, obj: str,
                       threshold: float = 0.5) -> Dict[str, object]:
    """"a {obj}" vs "not a {obj}" vs "no {obj}", on the segmenter.

    The detector version of this probe compares boxes. Here you get something
    sharper: a presence score. `presence_cost_of_not` is how much saying "not"
    moved the model's own belief that the thing is there. If that number is
    small, the text encoder did not process the negation -- and since this
    encoder is CLIP-family, that is the same root cause as in the detector.
    """
    out = {}
    for name, phrase in (("affirmed", f"a {obj}"), ("negated", f"not a {obj}"),
                         ("negated_alt", f"no {obj}")):
        insts = segmenter.segment(image, phrase, threshold)
        out[name] = {"phrase": phrase, "presence": segmenter.presence(image, phrase),
                     "n_instances": len(insts),
                     "top_score": insts[0].score if insts else 0.0}
    return {
        "obj": obj, "detail": out,
        "presence_cost_of_not": round(out["affirmed"]["presence"]
                                      - out["negated"]["presence"], 4),
        "still_finds_it_when_negated": out["negated"]["n_instances"] > 0,
    }


def affordance_probe(segmenter: Sam3Segmenter, image, nouns: Sequence[str],
                     affordances: Sequence[str], threshold: float = 0.5
                     ) -> Dict[str, object]:
    """Noun-shaped queries against purpose-shaped ones.

    A robot does not want "the sofa", it wants "somewhere to sit". If the nouns
    work and the affordances return nothing -- with a LOW presence score rather
    than an uncertain one -- then the model is not merely failing to find them,
    it is asserting they are absent. Any system that appears to answer
    affordance queries is doing that reasoning somewhere else.
    """
    def row(c):
        insts = segmenter.segment(image, c, threshold)
        return {"presence": segmenter.presence(image, c), "n_instances": len(insts),
                "top_score": insts[0].score if insts else 0.0}

    nouns, affordances = list(nouns), list(affordances)
    return {"nouns": {c: row(c) for c in nouns},
            "affordances": {c: row(c) for c in affordances}}


def sam3_vs_owlvit_sam(detector: OwlDetector, segmenter: Sam3Segmenter,
                       sam: SamSegmenter, image, phrase: str,
                       threshold: float = 0.1, mask_threshold: float = 0.5
                       ) -> Dict[str, object]:
    """text->instances (SAM 3) against text->box->pixels (OWL-ViT + SAM).

    Both pipelines now return instance masks, so this is a fair comparison of
    two ways to get from a word to pixels. SAM 3's mask is compared against the
    SAM mask grown from the detector's best box.

    Low agreement is not automatically SAM losing: SAM's boundary is only as
    good as the box it was handed, and that box came from the detector's
    language understanding. A perfect boundary around the wrong object is still
    the wrong object.
    """
    box, det_score = detector.top_box(image, phrase, threshold)
    s3 = segmenter.segment(image, phrase, mask_threshold)
    s3_top = s3[0].mask if s3 else np.zeros((image.height, image.width), bool)
    base = {
        "phrase": phrase, "detector_score": det_score,
        "sam3_n_instances": len(s3),
        "sam3_top_score": s3[0].score if s3 else 0.0,
        "sam3_presence": segmenter.presence(image, phrase),
        "sam3_area_fraction": round(float(s3_top.mean()), 4),
        "sam3_mask": s3_top,
    }
    if box is None:
        base.update({"detector_found_box": False, "sam_area_fraction": None,
                     "mask_agreement_iou": None, "sam_mask": None,
                     "note": "detector found no box above threshold, so SAM had "
                             "nothing to segment"})
        return base
    sam_mask = sam.mask_from_box(image, box)
    base.update({
        "detector_found_box": True, "detector_box": [round(v, 1) for v in box],
        "sam_area_fraction": round(float(sam_mask.mean()), 4),
        "mask_agreement_iou": mask_iou(s3_top, sam_mask), "sam_mask": sam_mask,
    })
    return base


# --------------------------------------------------------------------------- #
# Label-free consistency probes — VLM
# --------------------------------------------------------------------------- #


@dataclass
class VlmTrial:
    kind: str
    obj: str
    question: str
    raw: str
    parsed: Optional[str]


def phrasing_flip_probe(vlm, image, objs: Sequence[str]) -> Dict[str, object]:
    """The v1 probe: one object, four phrasings of the same yes/no question.

    A grounded model answers all four identically. `flip_rate` counts objects
    whose answer changed with wording alone.
    """
    trials, flipped = [], 0
    for obj in objs:
        answers = []
        for tmpl in QUESTION_TEMPLATES:
            q = tmpl.format(obj=obj)
            raw = vlm.ask(image, q)
            parsed = parse_yes_no(raw)
            answers.append(parsed)
            trials.append(VlmTrial("phrasing", obj, q, raw, parsed))
        if len(set(answers)) > 1:
            flipped += 1
    return {"n_objs": len(objs), "n_flipped": flipped,
            "flip_rate": round(flipped / max(1, len(objs)), 3), "trials": trials}


def negation_pair_probe(vlm, image, objs: Sequence[str]) -> Dict[str, object]:
    """"Is there a X?" and "Is there no X?" -- a logical pair.

    Coherent answers are (yes, no) or (no, yes). Getting the SAME answer to
    both is self-contradiction, and needs no ground truth to detect: you do not
    have to know whether X is there to know the model cannot have it both ways.
    """
    trials, bad, scored = [], 0, 0
    for obj in objs:
        q1 = f"Is there a {obj} in the image? Answer yes or no."
        q2 = f"Is there no {obj} in the image? Answer yes or no."
        r1, r2 = vlm.ask(image, q1), vlm.ask(image, q2)
        a1, a2 = parse_yes_no(r1), parse_yes_no(r2)
        trials += [VlmTrial("negation", obj, q1, r1, a1),
                   VlmTrial("negation", obj, q2, r2, a2)]
        if a1 is None or a2 is None:
            continue          # refusing to answer is not a contradiction
        scored += 1
        if a1 == a2:
            bad += 1
    return {"n_objs": len(objs), "n_scored": scored, "n_contradictions": bad,
            "contradiction_rate": round(bad / max(1, scored), 3), "trials": trials}


def spatial_pair_probe(vlm, image, pairs: Sequence[Tuple[str, str]],
                       relation: str = "to the left of") -> Dict[str, object]:
    """"Is A {relation} B?" and "Is B {relation} A?" -- mutually exclusive.

    Two yeses is impossible; two noes is possible only if the objects do not
    exist or genuinely do not stand in that relation, so it is counted
    separately rather than as an outright contradiction.
    """
    trials, both_yes, both_no, scored = [], 0, 0, 0
    for a, b in pairs:
        q1 = f"Is the {a} {relation} the {b}? Answer yes or no."
        q2 = f"Is the {b} {relation} the {a}? Answer yes or no."
        r1, r2 = vlm.ask(image, q1), vlm.ask(image, q2)
        p1, p2 = parse_yes_no(r1), parse_yes_no(r2)
        trials += [VlmTrial("spatial", f"{a}|{b}", q1, r1, p1),
                   VlmTrial("spatial", f"{a}|{b}", q2, r2, p2)]
        if p1 is None or p2 is None:
            continue
        scored += 1
        both_yes += (p1 == "yes" and p2 == "yes")
        both_no += (p1 == "no" and p2 == "no")
    return {"n_pairs": len(pairs), "n_scored": scored,
            "n_both_yes": both_yes, "n_both_no": both_no,
            "incoherence_rate": round(both_yes / max(1, scored), 3), "trials": trials}


# Markers that the model pushed back on a false premise instead of playing
# along. Word-boundary matched for the same reason as `parse_yes_no`: a bare
# substring test finds "no" inside "cannot" and "piano". Still deliberately
# simple and deliberately imperfect -- read the raw text before you trust the
# rate, and say in your report where the rule misjudged an answer.
REFUSAL_MARKERS = ("no", "none", "not", "zero", "cannot", "unable",
                   "n't", "never", "neither")
_REFUSAL_RE = re.compile(
    r"(\b(?:no|none|not|zero|cannot|unable|never|neither)\b|n't)")


def presupposition_probe(vlm, image, absent_objs: Sequence[str],
                         template: str = "How many {obj} are in the image?"
                         ) -> Dict[str, object]:
    """Leading questions about things that are not there.

    A yes/no question lets a model hedge; "How many X are there?" presupposes
    X exists and is a far stronger hallucination elicitor. `capitulation_rate`
    counts answers with no refusal marker -- i.e. the model answered the
    question as asked. ALWAYS read `trials` yourself; this keyword rule will
    misjudge some answers, and saying so in your report is part of the point.
    """
    trials, capitulated = [], 0
    for obj in absent_objs:
        q = template.format(obj=obj)
        raw = vlm.ask(image, q)
        refused = bool(_REFUSAL_RE.search(raw.lower()))
        capitulated += (not refused)
        trials.append(VlmTrial("presupposition", obj, q, raw,
                               "refused" if refused else "capitulated"))
    return {"n_objs": len(absent_objs), "n_capitulated": capitulated,
            "capitulation_rate": round(capitulated / max(1, len(absent_objs)), 3),
            "trials": trials}


# --------------------------------------------------------------------------- #
# Experiment log — turns a browsing session into report evidence
# --------------------------------------------------------------------------- #


@dataclass
class ExperimentLog:
    """Accumulates every probe you run so the report is not reconstructed from
    memory. `to_markdown()` output pastes straight into the deliverable."""

    rows: List[Dict[str, object]] = field(default_factory=list)

    def add(self, model: str, image_name: str, probe: str,
            prompt: str = "", note: str = "", **metrics) -> Dict[str, object]:
        row = {"model": model, "image": image_name, "probe": probe,
               "prompt": prompt, "note": note,
               "metrics": {k: v for k, v in metrics.items()
                           if isinstance(v, (int, float, str, bool, type(None)))}}
        self.rows.append(row)
        return row

    def to_markdown(self) -> str:
        """Long format: one row per (probe, metric).

        Wide format would give one sparse column per metric name across every
        probe -- twenty-odd columns, almost all blank, unusable in a 5-page
        report. Long format stays four columns wide however many probes you run.
        """
        if not self.rows:
            return "_(no experiments logged yet)_"
        head = "| model | image | probe | prompt | metric | value |"
        rule = "|---|---|---|---|---|---|"
        body = []
        for r in self.rows:
            metrics = r["metrics"] or {"(logged)": ""}
            for i, (k, v) in enumerate(metrics.items()):
                # Repeat the identifying columns only on the first metric row so
                # the table reads as grouped blocks rather than a wall.
                ident = ((str(r["model"]), str(r["image"]), str(r["probe"]),
                          str(r["prompt"])[:40]) if i == 0 else ("", "", "", ""))
                body.append("| " + " | ".join([*ident, str(k), str(v)]) + " |")
            if r.get("note"):
                body.append("| | | | | _note_ | " + str(r["note"]) + " |")
        return "\n".join([head, rule, *body])

    def to_json(self) -> str:
        return json.dumps(self.rows, indent=2, default=str)


# --------------------------------------------------------------------------- #
# Housekeeping
# --------------------------------------------------------------------------- #


def free_model(*objs) -> None:
    """Drop model references and return the VRAM.

    Call this between sections so a free-tier T4 never holds more models than
    the current section needs. Safe to call twice on the same object.

    Using a freed wrapper afterwards raises a clear RuntimeError telling you to
    re-create it (see `_Freeable`) -- without that, the next call died with a
    bare `no attribute 'processor'`, which is a confusing way to learn that a
    cell earlier in the notebook reclaimed your model.
    """
    import gc
    for o in objs:
        for attr in ("model", "processor"):
            if attr in getattr(o, "__dict__", {}):
                try:
                    delattr(o, attr)
                except Exception:  # noqa: BLE001
                    pass
        try:
            o._freed = True
        except Exception:  # noqa: BLE001
            pass
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:  # noqa: BLE001
        pass


def peak_vram_gb() -> Optional[float]:
    """Peak allocated VRAM since the last reset, in GB. None off-GPU."""
    try:
        import torch
        if torch.cuda.is_available():
            return round(torch.cuda.max_memory_allocated() / 1e9, 2)
    except Exception:  # noqa: BLE001
        pass
    return None


def vram_report(label: str = "", reset_peak: bool = False) -> Optional[Dict[str, float]]:
    """Print live and peak VRAM, so the T4 budget is visible rather than assumed.

    `max_memory_allocated` is a running maximum for the whole session, so the
    peak printed here is cumulative unless something reset it. That is what you
    want for a "does this fit on a T4" answer -- the number to compare against
    ~15 GB is the largest peak reached at any point, not the current occupancy.
    """
    try:
        import torch
        if not torch.cuda.is_available():
            print(f"[vram] {label}: no GPU (CPU run)")
            return None
        live = round(torch.cuda.memory_allocated() / 1e9, 2)
        peak = round(torch.cuda.max_memory_allocated() / 1e9, 2)
        print(f"[vram] {label:<28} live={live:>5.2f} GB   session peak={peak:>5.2f} GB")
        if reset_peak:
            torch.cuda.reset_peak_memory_stats()
        return {"live_gb": live, "peak_gb": peak}
    except Exception:  # noqa: BLE001
        return None


def load_images_from_dir(directory: str) -> Dict[str, "object"]:
    """Local/offline counterpart to the notebook's embedded image bank."""
    from PIL import Image
    out = {}
    for fname in sorted(os.listdir(directory)):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            out[os.path.splitext(fname)[0]] = Image.open(
                os.path.join(directory, fname)).convert("RGB")
    return out


# --------------------------------------------------------------------------- #
# Rendering — the point of the lab is to SEE the output, not read a dict
# --------------------------------------------------------------------------- #

# Colour-blind-safe qualitative palette (Okabe-Ito). One colour per prompt, so
# the same prompt keeps its colour across every figure in a section.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
           "#E69F00", "#56B4E9", "#F0E442", "#000000"]


def _finish(fig, suptitle: str = "") -> None:
    """Lay out and show. Reserves headroom for the suptitle -- a plain
    tight_layout() lets per-panel titles collide with it."""
    import matplotlib.pyplot as plt
    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
        fig.tight_layout(rect=(0, 0, 1, 0.94))
    else:
        fig.tight_layout()
    plt.show()


def prompt_colors(prompts: Sequence[str]) -> Dict[str, str]:
    return {p: PALETTE[i % len(PALETTE)] for i, p in enumerate(prompts)}


def draw_detections(image, dets: Sequence[Det], title: str = "",
                    colors: Optional[Dict[str, str]] = None, ax=None,
                    max_boxes: int = 25, show_scores: bool = True):
    """Draw boxes with `prompt score` labels. Returns the axis.

    Labels sit inside the top edge of the box when the box starts near y=0, so
    text never falls off the top of the figure.
    """
    import matplotlib.pyplot as plt
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7 * image.height / image.width))
    ax.imshow(image)
    ax.set_axis_off()
    colors = colors or prompt_colors(sorted({d.prompt for d in dets}))
    for d in list(dets)[:max_boxes]:
        x0, y0, x1, y1 = d.box
        c = colors.get(d.prompt, PALETTE[0])
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                   edgecolor=c, linewidth=2.2))
        label = f"{d.prompt} {d.score:.2f}" if show_scores else d.prompt
        inside = y0 < 18
        ax.text(x0 + 2, y0 + (14 if inside else -5), label,
                fontsize=8.5, color="white", va="top" if inside else "bottom",
                bbox=dict(facecolor=c, edgecolor="none", pad=1.4, alpha=0.92))
    n = len(dets)
    suffix = f" — {n} box{'es' if n != 1 else ''}" + (
        f" (showing {max_boxes})" if n > max_boxes else "")
    ax.set_title((title or "detections") + suffix, fontsize=10)
    return ax


def show_detection_comparison(image, panels: Sequence[Tuple[str, Sequence[Det]]],
                              suptitle: str = "", max_boxes: int = 8):
    """Same image, one panel per prompt variant, shared colour per prompt.

    `max_boxes` caps what is DRAWN, not what was found -- at a low threshold a
    panel can hold 40+ overlapping boxes and become unreadable. Each panel title
    always reports the true count, and says so when it is showing fewer.
    """
    import matplotlib.pyplot as plt
    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 5.2 * image.height / image.width))
    axes = np.atleast_1d(axes)
    colors = prompt_colors(sorted({d.prompt for _, dets in panels for d in dets}))
    for ax, (title, dets) in zip(axes, panels):
        draw_detections(image, dets, title=title, colors=colors, ax=ax,
                        max_boxes=max_boxes)
    _finish(fig, suptitle)


def show_mask(image, mask_or_prob, title: str = "", ax=None,
              threshold: Optional[float] = None, cmap: str = "magma"):
    """Overlay a mask (boolean) or probability map (float) on the image."""
    import matplotlib.pyplot as plt
    arr = np.asarray(mask_or_prob)
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6 * image.height / image.width))
    ax.imshow(image)
    if arr.dtype == bool:
        overlay = np.zeros((*arr.shape, 4))
        overlay[arr] = [0.0, 0.45, 0.70, 0.55]      # Okabe-Ito blue at 55%
        ax.imshow(overlay)
    else:
        ax.imshow(arr, cmap=cmap, alpha=0.55, vmin=0, vmax=1)
        if threshold is not None:
            ax.contour(arr, levels=[threshold], colors="#00FFC8", linewidths=1.8)
    ax.set_axis_off()
    ax.set_title(title, fontsize=10)
    return ax


def show_mask_grid(image, named_masks: Dict[str, np.ndarray], suptitle: str = "",
                   threshold: Optional[float] = None, ncols: int = 3):
    """One panel per prompt. Titles carry the masked-area fraction, because the
    visual size of a mask is hard to compare by eye across panels."""
    import matplotlib.pyplot as plt
    items = list(named_masks.items())
    ncols = min(ncols, max(1, len(items)))
    nrows = int(np.ceil(len(items) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(4.6 * ncols, 4.6 * nrows * image.height / image.width))
    axes = np.atleast_1d(axes).ravel()
    for ax, (name, m) in zip(axes, items):
        arr = np.asarray(m)
        frac = float((arr > (threshold if threshold is not None else 0.5)).mean()
                     if arr.dtype != bool else arr.mean())
        show_mask(image, arr, title=f"{name}\narea={frac:.1%}", ax=ax, threshold=threshold)
    for ax in axes[len(items):]:
        ax.set_axis_off()
    _finish(fig, suptitle)


def draw_instances(image, instances: Sequence[Instance], title: str = "",
                   ax=None, show_boxes: bool = True):
    """Overlay each instance in its own colour, with its score.

    Instance segmentation is the whole point of this head, so the figure has to
    make separate instances visually separate -- a single merged blob would
    render the same as a semantic mask and hide what changed.
    """
    import matplotlib.pyplot as plt
    from matplotlib.colors import to_rgb
    if ax is None:
        _, ax = plt.subplots(figsize=(6.6, 6.6 * image.height / image.width))
    ax.imshow(image)
    ax.set_axis_off()
    overlay = np.zeros((image.height, image.width, 4))
    for k, inst in enumerate(instances):
        rgb = to_rgb(PALETTE[k % len(PALETTE)])
        overlay[inst.mask] = [*rgb, 0.55]
        x0, y0, x1, y1 = inst.box
        if show_boxes:
            ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                       edgecolor=rgb, linewidth=2.0))
        inside = y0 < 18
        ax.text(x0 + 2, y0 + (14 if inside else -5), f"{inst.concept} {inst.score:.2f}",
                fontsize=8.5, color="white", va="top" if inside else "bottom",
                bbox=dict(facecolor=rgb, edgecolor="none", pad=1.4, alpha=0.92))
    if len(instances):
        ax.imshow(overlay)
    n = len(instances)
    ax.set_title(f"{title or 'instances'} — {n} instance{'s' if n != 1 else ''}"
                 + (" (nothing above threshold)" if n == 0 else ""), fontsize=10)
    return ax


def show_instance_comparison(image, panels: Sequence[Tuple[str, Sequence[Instance]]],
                             suptitle: str = ""):
    """One panel per concept / condition, instances coloured separately."""
    import matplotlib.pyplot as plt
    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 5.2 * image.height / image.width))
    axes = np.atleast_1d(axes)
    for ax, (title, insts) in zip(axes, panels):
        draw_instances(image, insts, title=title, ax=ax)
    _finish(fig, suptitle)


def plot_metric_bars(values: Dict[str, float], title: str, ylabel: str = "",
                     ylim: Optional[Tuple[float, float]] = (0, 1),
                     color: str = "#0072B2", annotate: bool = True):
    """Small labelled bar chart. Used for every summary number in the lab so the
    figures in a student report look like one family."""
    import matplotlib.pyplot as plt
    keys = list(values)
    vals = [values[k] for k in keys]
    fig, ax = plt.subplots(figsize=(max(4.5, 1.5 * len(keys)), 3.6))
    bars = ax.bar(range(len(keys)), vals, color=color)
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels(keys, rotation=20, ha="right", fontsize=9)
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel or "value", fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
    if annotate:
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height(),
                    f"{v:.2f}", ha="center", va="bottom", fontsize=8.5)
    fig.tight_layout()
    plt.show()


def plot_threshold_sweep(rows: Sequence[Dict[str, object]], x: str, y: str,
                         title: str, ylabel: str = ""):
    """Line plot for the two sweep helpers (box count / mask area vs threshold)."""
    import matplotlib.pyplot as plt
    xs = [r[x] for r in rows]
    ys = [r[y] for r in rows]
    fig, ax = plt.subplots(figsize=(5.4, 3.6))
    ax.plot(xs, ys, marker="o", color="#D55E00")
    for xv, yv in zip(xs, ys):
        ax.annotate(f"{yv:g}", (xv, yv), textcoords="offset points",
                    xytext=(0, 7), ha="center", fontsize=8)
    ax.set_xlabel(x.replace("_", " "), fontsize=9)
    ax.set_ylabel(ylabel or y.replace("_", " "), fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.25)
    ax.set_axisbelow(True)
    fig.tight_layout()
    plt.show()


def show_vlm_trials(trials: Sequence[VlmTrial], limit: int = 8) -> None:
    """Print the raw model text next to the parsed verdict. The raw text is the
    evidence; the parsed label is a convenience that can be wrong."""
    print(f"{'kind':<15} {'object':<22} {'parsed':<11} raw answer")
    print("-" * 96)
    for t in list(trials)[:limit]:
        print(f"{t.kind:<15} {t.obj[:21]:<22} {str(t.parsed):<11} {t.raw[:48]!r}")
    if len(trials) > limit:
        print(f"... {len(trials) - limit} more (inspect the `trials` list directly)")


print("lab core loaded — 50 helpers available")

In [ ]:
#@title Starter image bank — 6 photos, embedded (run me) { display-mode: "form" }
# Embedded rather than downloaded so the lab is reproducible years from now and
# works with no network. All six are CC0 or Public Domain Mark; provenance is
# printed below and must be reproduced if you republish a figure.
import base64, io, json
from PIL import Image

_BANK = json.loads(r"""{"cat": "...", "coffee_mug": "...", "fruit_bowl": "...", "desk": "...", "living_room": "...", "road_sign": "..."}""")
IMAGES = {k: Image.open(io.BytesIO(base64.b64decode(v))).convert("RGB")
          for k, v in _BANK.items()}

IMAGE_CREDITS = [
    ("living_room", "living room sofa, television, armchair", "unknown", "cc0 1.0", "https://www.rawpixel.com/image/3285035/free-photo-image-living-room-map"),
    ("coffee_mug", "Black coffee in a retro mug", "freestocks.org", "cc0 1.0", "https://www.flickr.com/photos/135396164@N05/44381116112"),
    ("desk", "Macbook Laptop", "Pawel Kadysz", "cc0 1.0", "https://stocksnap.io/photo/macbook-laptop-R66E2T133W"),
    ("road_sign", "One Lane Traffic - Construction", "GlacierNPS", "pdm 1.0", "https://www.flickr.com/photos/43288043@N04/52144262918"),
    ("fruit_bowl", "Fruits Oranges", "JESHOOTS.com", "cc0 1.0", "https://stocksnap.io/photo/fruits-oranges-975AABA4B2"),
    ("cat", "Ellie cat on the living room sofa", "weremonkey", "pdm 1.0", "https://www.flickr.com/photos/44642876@N00/37472932036"),
]

print(f"{len(IMAGES)} starter images:")
for _n, _im in IMAGES.items():
    print(f"  {_n:<12} {_im.size}")
print("\nProvenance (all CC0 / Public Domain Mark):")
for _n, _t, _c, _l, _u in IMAGE_CREDITS:
    print(f"  {_n:<12} {_l:<8} {str(_t)[:38]:<40} {_c}")
    print(f"  {'':<12} {_u}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (name, im) in zip(axes.ravel(), IMAGES.items()):
    ax.imshow(im); ax.set_axis_off(); ax.set_title(f"{name}  {im.size}", fontsize=10)
fig.suptitle("Starter images — pick targets from these, then bring your own", fontsize=13)
fig.tight_layout(); plt.show()

log = ExperimentLog()   # every probe you care about goes in here for the report

---
# §1 — OWL-ViT: text in, boxes out

`google/owlvit-base-patch32`. You hand it a **list of text queries** and it
scores every candidate box against every query. There is no fixed class list —
"a yellow armchair" is as valid a query as "a chair".

Start with the easy case so you know what *working* looks like.

In [ ]:
det = OwlDetector(device=DEVICE)          # ~600 MB
print("loaded:", det.model_id)
vram_report("after OWL-ViT load")

In [ ]:
img = IMAGES["cat"]
dets = det.detect(img, ["a cat", "a window", "a cushion"], threshold=0.1)
draw_detections(img, dets, title="OWL-ViT · threshold=0.1")
plt.show()
for d in dets:
    print(f"  {d.prompt:<12} {d.score:.3f}  box={[round(v) for v in d.box]}")

### Now the threshold you did not think about

`threshold=0.1` was a number in the call above, not a property of the model.
Every "number of objects in this image" a detector reports is a cut *you* chose.

In [ ]:
img = IMAGES["desk"]
sweep = threshold_sweep(det, img, ["a keyboard", "a laptop"])
plot_threshold_sweep(sweep, "threshold", "n_boxes",
                     "How many boxes exist? Depends on your cut.", "boxes returned")
show_detection_comparison(img, [
    (f"threshold={t}", det.detect(img, ["a keyboard", "a laptop"], threshold=t))
    for t in (0.02, 0.1, 0.3)
], suptitle="Same image, same prompts, three thresholds")

**Checkpoint.** Write down the box count at 0.02 and at 0.3 before moving on.
If a system card says "detects 29 objects", you now know that sentence is
incomplete without the threshold.

---
### 1.1 Does the wording matter? (template sensitivity)

Same object, four ways of naming it. A grounded detector should not care.
`mean_pairwise_iou` = 1.0 means the box never moved; `score_spread` is how much
the *confidence* swung.

In [ ]:
img = IMAGES["cat"]
phrases = [t.format(obj="cat") for t in PROMPT_TEMPLATES]
r = phrase_set_consistency(det, img, phrases, threshold=0.1)
print("phrases      :", r["phrases"])
print("scores       :", r["scores"])
print("mean box IoU :", r["mean_pairwise_iou"], " score spread:", r["score_spread"])
plot_metric_bars(r["scores"], "Same cat, four phrasings — OWL-ViT confidence",
                 ylabel="top score")
log.add("owlvit", "cat", "template_sensitivity", prompt="cat x4",
        mean_pairwise_iou=r["mean_pairwise_iou"], score_spread=r["score_spread"])

Look at the bare noun. Dropping one article is not a paraphrase to a human;
to this model it can be the difference between a confident detection and
nothing at all. **Whatever you measured, record it** — that is your number, not
a number from a lecture slide.

---
### 1.2 Synonyms — same thing, different word

In [ ]:
for name, a, b in [("coffee_mug", "a mug", "a cup"),
                   ("desk", "a laptop", "a notebook"),
                   ("living_room", "a couch", "a sofa")]:
    r = synonym_stability(det, IMAGES[name], a, b, threshold=0.1)
    print(f"{name:<12} {a!r} vs {b!r}: IoU={r['top_box_iou']:<5} "
          f"scores={r['score_a']}/{r['score_b']} delta={r['score_delta']}")
    log.add("owlvit", name, "synonym", prompt=f"{a} vs {b}",
            top_box_iou=r["top_box_iou"], score_delta=r["score_delta"])

img = IMAGES["desk"]
show_detection_comparison(img, [
    ("a laptop", det.detect(img, ["a laptop"], threshold=0.1)),
    ("a notebook", det.detect(img, ["a notebook"], threshold=0.1)),
], suptitle="Two words a thesaurus calls synonyms")

Two different outcomes worth separating in your notes:

* **Same box, different score** — the model found the right thing but is much
  less sure it has the right *word*. Nothing is wrong with the grounding; the
  confidence number is just not comparable across queries.
* **Different box entirely** — the phrase pointed somewhere else. "Notebook" is
  genuinely ambiguous (laptop? paper notebook?), and the model resolved it its
  own way, silently.

---
### 1.3 Up and down the ladder — can it do categories?

Superordinate ("an animal"), basic ("a cat"), subordinate ("a kitten").
Humans move up and down this ladder effortlessly.

In [ ]:
for name, ladder in [("cat", ["an animal", "a pet", "a cat", "a kitten"]),
                     ("fruit_bowl", ["fruit", "citrus", "a lime", "a key lime"]),
                     ("road_sign", ["a vehicle", "a car", "a truck", "a pickup truck"])]:
    scores = det.max_scores(IMAGES[name], ladder)
    print(f"{name:<12} {scores}")
    log.add("owlvit", name, "hypernym_ladder", prompt=" > ".join(ladder),
            **{f"score_{i}": v for i, v in enumerate(scores.values())})

plot_metric_bars(det.max_scores(IMAGES["cat"], ["an animal", "a pet", "a cat", "a kitten"]),
                 "Is a cat an animal? — OWL-ViT, one clear cat photo",
                 ylabel="best score (no threshold)")

This is the first result that should genuinely bother you. Note also what the
road scene does with **"a vehicle"** versus **"a car"** on a photo whose
foreground vehicle is a pickup truck.

Write one sentence for your report: if a robot's task specification says *"pick
up the vehicle"* or *"avoid animals"*, what does this measurement predict?

---
### 1.4 Attribute binding — the colour and the shape must go *together*

The v1 lab did this with synthetic shapes. Same trap, real photos: ask for a
combination whose colour and object are both present in the scene but not
attached to each other.

In [ ]:
print("fruit_bowl — TRUE: a red apple, a green lime.  TRAPS: green apple, red lime")
print("  ", det.max_scores(IMAGES["fruit_bowl"],
                           ["a red apple", "a green lime", "a green apple", "a red lime"]))
print("living_room — TRUE: a grey sofa, a yellow chair.  TRAPS: yellow sofa, grey chair")
print("  ", det.max_scores(IMAGES["living_room"],
                           ["a grey sofa", "a yellow chair", "a yellow sofa", "a grey chair"]))
print("coffee_mug — a light-blue enamel mug holding black coffee (deliberately ambiguous)")
print("  ", det.max_scores(IMAGES["coffee_mug"],
                           ["a blue mug", "a black mug", "black coffee", "blue coffee"]))

Compare each **true** pairing against its **trap**. Where the trap beats the
truth, the model is matching a bag of words — the colour and the noun were both
in the image, so the phrase scored well, and nothing checked that they belonged
to the same object. Look especially at whether the *true* pairing for the apple
outscores the trap built from the lime's colour.

The mug is the case to argue about rather than score. It is a blue mug
containing black coffee, so "a black mug" is not straightforwardly wrong — a
human would hesitate too. Decide for yourself whether that counts as a binding
failure, and say which of your other results made you decide that way. Some of
what looks like model error is really **specification** error: the prompt did
not say what you meant.

---
### 1.5 Negation — is there an operator for "not"?

In [ ]:
img = IMAGES["desk"]
r = negation_probe(det, img, "keyboard", threshold=0.1)
for k, v in r["detail"].items():
    print(f"  {k:<12} {v['phrase']!r:<20} score={v['score']}")
print(f"  IoU('a keyboard', 'not a keyboard') = {r['iou_affirm_vs_negated']}")
print(f"  IoU('a keyboard', 'no keyboard')    = {r['iou_affirm_vs_negated_alt']}")

show_detection_comparison(img, [
    ("a keyboard", det.detect(img, ["a keyboard"], threshold=0.1)),
    ("not a keyboard", det.detect(img, ["not a keyboard"], threshold=0.1)),
], suptitle="Does the word 'not' do anything?")
log.add("owlvit", "desk", "negation", prompt="a/not a keyboard",
        iou_affirm_vs_negated=r["iou_affirm_vs_negated"])

A CLIP-style text encoder has no logical operator. "Not a keyboard" is a
sentence whose *embedding* still sits near keyboards. Try `"a cat"` /
`"not a cat"` on the cat image too — you may find the behaviour is not even
consistent between phrasings of the negation, which is worse than a clean
failure, because it makes the failure hard to test for.

---
### 1.6 Absent objects — the detector cannot say "nothing"

There is no abstain output. Ask for a giraffe and you get a ranked box; the only
question is what score it carries. So the real question is whether the
absent-object scores are **separable** from the present-object ones — because
if they overlap, no threshold you pick can tell them apart.

In [ ]:
r = absent_object_probe(det, IMAGES["fruit_bowl"],
                        absent=["a lemon", "a grapefruit", "a car"],
                        present=["a lime", "an orange", "a banana"])
print("ABSENT :", r["absent_scores"])
print("PRESENT:", r["present_scores"])
print("max absent =", r["max_absent_score"], " min present =", r["min_present_score"])
print("separation margin =", r["separation_margin"],
      "  <-- negative means NO threshold separates present from absent here")
plot_metric_bars({**r["absent_scores"], **r["present_scores"]},
                 "Absent (first) vs present (last) — is there a clean cut?",
                 ylabel="best score")
log.add("owlvit", "fruit_bowl", "absent_separation",
        prompt="lemon/grapefruit/car vs lime/orange/banana",
        max_absent=r["max_absent_score"], min_present=r["min_present_score"],
        separation_margin=r["separation_margin"])

Note *which* absent object scores highest, and how plausible it is in that
scene. A lemon in a fruit bowl is not a random absent object — it is exactly
what a model with a strong contextual prior would expect to be there. Keep this
result; you will meet the same object again in §5 and §6.

---
### 1.7 The one that surprises everyone: your word list changes the labels

OWL-ViT scores all 576 candidate boxes against every query, then reports each
box under its **single best-matching** query. So the label a region gets depends
on which *other* words you happened to ask about.

In [ ]:
r = det.vocabulary_competition(IMAGES["fruit_bowl"], target="an orange",
                               competitor="a lemon", threshold=0.02)
print(f"'an orange' alone      : {r['n_target_alone']} boxes, top score {r['score_target_alone']}")
print(f"'a lemon'   alone      : top score {r['score_competitor_alone']}")
print(f"'an orange' when asked alongside 'a lemon': {r['n_target_together']} boxes")
print(f"boxes taken over by 'a lemon': {r['boxes_lost_by_target']}")

show_detection_comparison(IMAGES["fruit_bowl"], [
    ("asked: ['an orange']", r["dets_alone"]),
    ("asked: ['an orange', 'a lemon']", r["dets_together"]),
], suptitle="Same image. Same model. One extra word in the query list.",
   max_boxes=5)   # top 5 only; titles report the true counts
log.add("owlvit", "fruit_bowl", "vocab_competition", prompt="an orange vs a lemon",
        n_alone=r["n_target_alone"], n_together=r["n_target_together"],
        boxes_lost=r["boxes_lost_by_target"])

Neither score changed. The image did not change. **The vocabulary changed**, and
regions were reassigned to a word for a fruit that is not in the bowl.

For a robot with an open-vocabulary perception stack, this means the object list
you get back is a function of the prompt list someone configured — add a word
for an object that never appears and you can silently relabel things that do.

---
# §2 — Your turn: any image, any prompts

Upload your own photos and probe them. **This is where the deliverable comes
from** — the starter images are a warm-up.

Run the upload cell (or skip it and keep using `IMAGES`), then edit `TARGET` and
`PROMPTS` in the next cell and re-run it as many times as you like.

In [ ]:
# Upload your own images (Colab). Skip this cell to work with the starter set.
try:
    from google.colab import files
    uploaded = files.upload()          # opens a file picker
    import io as _io
    from PIL import Image as _Image
    for fname, data in uploaded.items():
        key = fname.rsplit(".", 1)[0]
        IMAGES[key] = _Image.open(_io.BytesIO(data)).convert("RGB")
        print("added:", key, IMAGES[key].size)
except ImportError:
    print("Not on Colab. Add images with:")
    print("   IMAGES['myphoto'] = Image.open('/path/to/photo.jpg').convert('RGB')")
print("\navailable:", list(IMAGES))

In [ ]:
# ==== EDIT ME ================================================================
TARGET    = "living_room"        # any key in IMAGES
PROMPTS   = ["a sofa", "a yellow chair", "a lamp", "a framed map"]
THRESHOLD = 0.1
# =============================================================================

img = IMAGES[TARGET]
dets = det.detect(img, PROMPTS, threshold=THRESHOLD)
draw_detections(img, dets, title=f"{TARGET} · threshold={THRESHOLD}")
plt.show()
print("max score per prompt (each queried on its own, no threshold):")
for p, s in det.max_scores(img, PROMPTS).items():
    print(f"   {p:<28} {s}")

In [ ]:
# Found something worth reporting? Log it, with a note in your own words.
log.add("owlvit", TARGET, "free_exploration", prompt=str(PROMPTS),
        n_boxes=len(dets), top_score=dets[0].score if dets else 0.0,
        note="EDIT ME: what surprised you?")
print(log.to_markdown())

---
# §3 — SAM 3: text in, *instances* out

A box says "the mug is somewhere in this rectangle". A robot that has to grasp
the mug, or decide which floor pixels are safe to drive over, needs to know
**which pixels** — and if there are three keyboards on the desk, it needs them
as three separate things, not one keyboard-coloured blob.

This model gives you two outputs, and you should measure both:

* **instances** — a mask, a box and a score per occurrence of the concept
* **presence** — one number for *"is this concept in the image at all?"*

That second output is new. OWL-ViT and CLIP-style segmenters have no way to say
"nothing here" — they only score low. SAM 3 has a head whose entire job is that
question, so for the first time in this lab you can ask a model whether it
knows something is absent.

### Read this before you cite it

**This is not the lecture's model.** The lecture teaches **SegCLIP** (Luo et al.,
arXiv 2211.14813) — annotation-free, open-vocabulary, *semantic* segmentation.
Its checkpoint needs torch 1.8 / mmcv-full 1.3.14 / Python 3.8 and **will not
install on current Colab**. This lab runs a **SAM 3** variant instead, which is a
different architecture class (an instance decoder trained with mask supervision
at scale). Nothing you measure here speaks to SegCLIP's annotation-free claim.
If you write about SegCLIP, say which model you actually ran.

**The checkpoint is third-party and its licence is unsettled.**
`vil-uob/sam3-litetext-s0` is SAM 3 with its text encoder distilled down to a
MobileCLIP-based one (~88 % fewer text-encoder parameters), released by the
Visual Information Lab — *not* by Meta. Its card says Apache-2.0, but the
weights derive from SAM 3, whose official release (`facebook/sam3`) is gated
under Meta's own licence. Treat the Apache-2.0 label as possibly mistaken. This
notebook downloads at runtime and redistributes nothing.

**Why it is still the right subject.** Its text encoder is *MobileCLIP* — still
CLIP-family. So when negation and superordinate categories fail here in exactly
the way they failed in §1, you can name the mechanism instead of just noting the
coincidence: it is the same kind of text encoder, and the failure travels with
it.

In [ ]:
# Release the detector before loading the segmenter. On a free T4 the budget is
# ~15 GB and this notebook drives four models, so each section hands back what
# it no longer needs. Using `det` after this raises a readable error telling you
# to re-create it -- §4 does exactly that.
free_model(det)
vram_report("after freeing OWL-ViT")

seg = Sam3Segmenter(device=DEVICE)           # 529 M params
print("loaded:", seg.model_id)
vram_report("after SAM 3 load")

In [ ]:
img = IMAGES["desk"]                         # three keyboards are in this frame
insts = seg.segment(img, "keyboard", threshold=0.5)
draw_instances(img, insts, title="SAM 3 · 'keyboard'")
plt.show()
for i in insts:
    print(f"  {i.concept:<12} score={i.score:<6} area={i.mask.mean():.3f}  "
          f"box={[round(v) for v in i.box]}")
print("presence('keyboard') =", seg.presence(img, "keyboard"))

Three keyboards, three masks, three scores. Compare that with what a *semantic*
segmenter would have given you: one region covering all keyboard-like pixels,
with no way to ask "how many?" — the question a robot that must pick one of them
actually has.

---
### 3.1 The presence head — can a model say "nothing here"?

Ask for concepts that are in the image and concepts that are not. The number to
watch is `separation_margin`: lowest present score minus highest absent score.
**Positive means some threshold separates them. Negative means none can.**

In [ ]:
img = IMAGES["fruit_bowl"]     # a lime, an orange, an apple, a banana. No lemon.
r = presence_probe(seg, img,
                   present=["lime", "orange", "apple", "banana"],
                   absent=["lemon", "grapefruit", "car"])
print("PRESENT:", r["present_presence"])
print("ABSENT :", r["absent_presence"])
print(f"min present={r['min_present']}  max absent={r['max_absent']}  "
      f"separation margin={r['separation_margin']}")
plot_metric_bars({**r["present_presence"], **r["absent_presence"]},
                 "Presence score — present concepts (first) vs absent (last)",
                 ylabel="P(concept is in image)")
log.add("sam3", "fruit_bowl", "presence_separation",
        prompt="lime/orange/apple/banana vs lemon/grapefruit/car",
        min_present=r["min_present"], max_absent=r["max_absent"],
        separation_margin=r["separation_margin"])

Now add a **superordinate** word to the present list and re-run in your head:
where does `citrus` land relative to `lemon`? One of those is in the bowl and
one is not.

In [ ]:
for c in ["lime", "citrus", "fruit", "lemon", "grapefruit", "car"]:
    print(f"  presence({c!r:<14}) = {seg.presence(IMAGES['fruit_bowl'], c)}")

Write down the gap between the *present* superordinate and the *absent* fruit.
If a word for something in the bowl scores no higher than a word for something
that is not, then this head cannot be used as an "is it there" gate — however
well it does on the easy cases.

### 3.2 Abstention, or just a threshold?

Some concepts return zero instances. That looks like the model refusing — the
thing the older models could never do. Test whether it is refusal or arithmetic:
lower the cut and see whether candidates were there all along.

In [ ]:
print("cat photo, asked for 'dog':")
for row in score_threshold_sweep(seg, IMAGES["cat"], "dog"):
    print("   ", row)
print("\nfruit bowl (no lemon), asked for 'lemon':")
sweep = score_threshold_sweep(seg, IMAGES["fruit_bowl"], "lemon")
for row in sweep:
    print("   ", row)
plot_threshold_sweep(sweep, "threshold", "n_instances",
                     "'lemon' on a bowl with no lemon — instances vs your cut",
                     "instances returned")
log.add("sam3", "fruit_bowl", "abstention_is_threshold", prompt="lemon",
        n_at_0p05=sweep[0]["n_instances"], n_at_0p5=sweep[-2]["n_instances"])

In [ ]:
show_instance_comparison(IMAGES["fruit_bowl"], [
    (f"'lemon' @ thr={t}", seg.segment(IMAGES["fruit_bowl"], "lemon", threshold=t))
    for t in (0.05, 0.3, 0.5)
], suptitle="Same model, same absent fruit, three thresholds")

Two different behaviours to distinguish and record:

* One of those concepts stays empty **all the way down**. The model genuinely
  has nothing to offer — real abstention.
* The other produces candidates as soon as you relax the cut. Nothing was
  refused; the **default threshold** did the refusing, and it happens to be
  hiding masks over fruit that is really there.

This matters more than it looks. A perception stack that reports "no lemon" is
making a much stronger claim than one that reports "no lemon above 0.5", and only
one of those is what the model actually said.

### 3.3 Phrasing — and a wobble a box probe cannot see

Same object, four phrasings. Now there are **two** ways for the answer to move:
the pixels can change, or the pixels can stay and be split into a different
number of instances.

In [ ]:
for name, obj in [("cat", "cat"), ("desk", "keyboard")]:
    phrases = [t.format(obj=obj) for t in PROMPT_TEMPLATES]
    r = instance_count_consistency(seg, IMAGES[name], phrases)
    print(f"{name:<8} counts={r['n_instances']}")
    print(f"{'':<8} presence={r['presence']}")
    print(f"{'':<8} mean mask IoU={r['mean_pairwise_iou']}  "
          f"count spread={r['count_spread']}  area spread={r['area_spread']}")
    log.add("sam3", name, "phrase_consistency", prompt=f"{obj} x4",
            mean_pairwise_iou=r["mean_pairwise_iou"], count_spread=r["count_spread"])

Compare `score_spread` from OWL-ViT in §1.1 against `mean_pairwise_iou` and
`count_spread` here. Same rewording, two heads — which one was steadier, and
does that tell you about grounding or about the head?

Look specifically at what **`"a photo of a X"`** does. In §1.1 that template was
the *best* phrasing for OWL-ViT — CLIP was trained on caption-like text, so
dressing the noun up as a caption helped. Check what it does to the presence
score here. If a phrasing that improves one CLIP-family model quietly switches
another one off, then "prompt engineering" is not a portable skill you learn
once: it is per-model, and a phrasing your interface hard-codes can survive a
model upgrade while silently ceasing to work.

### 3.4 Negation — the same failure, now with a number on it

`presence_cost_of_not` is how far the word "not" moved the model's own belief
that the thing is present. If it barely moves, the text encoder never processed
the negation.

In [ ]:
for name, obj in [("cat", "cat"), ("desk", "keyboard")]:
    r = negation_probe_seg(seg, IMAGES[name], obj)
    for k, v in r["detail"].items():
        print(f"  {name:<8} {v['phrase']!r:<20} presence={v['presence']:<8} "
              f"instances={v['n_instances']}")
    print(f"  {'':<8} presence cost of 'not' = {r['presence_cost_of_not']}   "
          f"still finds it when negated: {r['still_finds_it_when_negated']}\n")
    log.add("sam3", name, "negation", prompt=f"a/not a {obj}",
            presence_cost_of_not=r["presence_cost_of_not"])

img = IMAGES["desk"]
show_instance_comparison(img, [
    ("'a keyboard'", seg.segment(img, "a keyboard")),
    ("'not a keyboard'", seg.segment(img, "not a keyboard")),
    ("'no keyboard'", seg.segment(img, "no keyboard")),
], suptitle="Three ways of saying it is not there")

Note that the two negation phrasings may not behave the same way. An
inconsistent failure is worse than a clean one: you cannot write a test for it
without knowing which form the user will type.

### 3.5 Superordinates — the §1 failure, re-measured

OWL-ViT scored `"an animal"` near zero on an unmistakable cat. Does a 2025-era
model fix that?

In [ ]:
for name, ladder in [("cat", ["cat", "animal", "pet", "kitten"]),
                     ("fruit_bowl", ["lime", "citrus", "fruit"]),
                     ("road_sign", ["truck", "vehicle", "car"])]:
    scores = {c: seg.presence(IMAGES[name], c) for c in ladder}
    counts = {c: len(seg.segment(IMAGES[name], c)) for c in ladder}
    print(f"{name:<12} presence={scores}")
    print(f"{'':<12} instances={counts}")

plot_metric_bars({c: seg.presence(IMAGES["cat"], c)
                  for c in ["cat", "kitten", "pet", "animal"]},
                 "Is a cat an animal? — SAM 3 presence head", ylabel="presence")

Some of these it fixes and some it does not. **Both halves belong in your
report**, because "newer model, same failure" and "newer model, failure gone"
support opposite conclusions about whether the limitation is fundamental.

### 3.6 Parts and wholes

In [ ]:
for name, part, whole in [("cat", "ear", "cat"), ("coffee_mug", "handle", "mug"),
                          ("desk", "key", "keyboard")]:
    r = part_whole_probe(seg, IMAGES[name], part, whole)
    print(f"{name:<12} {part!r:<10} in {whole!r:<10} "
          f"containment={r['containment_part_in_whole']:<6} "
          f"area_ratio={r['area_ratio_part_over_whole']:<7} "
          f"presence: {r['presence_part']} / {r['presence_whole']}")
    log.add("sam3", name, "part_whole", prompt=f"{part} in {whole}",
            containment=r["containment_part_in_whole"],
            area_ratio=r["area_ratio_part_over_whole"],
            presence_part=r["presence_part"])

img = IMAGES["cat"]
show_instance_comparison(img, [("'ear'", seg.segment(img, "ear")),
                               ("'cat'", seg.segment(img, "cat"))],
                         suptitle="A part and its whole")

Watch for a part that scores **present** but returns **no instances**. The model
believes it is there and will not say where — which brings us to the next probe.

### 3.7 When one model disagrees with itself

In [ ]:
for name, concepts in [("desk", ["keyboard", "key", "notebook"]),
                       ("cat", ["cat", "whisker", "dog"]),
                       ("fruit_bowl", ["lime", "lemon", "citrus"])]:
    for c in concepts:
        r = presence_vs_instances(seg, IMAGES[name], c)
        flag = "  <-- DISAGREES" if r["disagrees"] else ""
        print(f"  {name:<12} {c:<12} presence={r['presence']:<8} "
              f"instances={r['n_instances']}  top={r['top_instance_score']}{flag}")

`presence > 0.5` with **zero** instances means the two heads of one model give
different answers to the same question. Whether that is a bug depends entirely
on which output your system reads — and a pipeline that only consumes masks
throws away what the model actually believed.

### 3.8 Can you ask it where to *do* something?

The question that matters for embodied AI. A robot does not want "the sofa" — it
wants "somewhere to sit", "free space to drive", "somewhere to put this mug".
Those are **affordances**, not nouns.

In [ ]:
for name, nouns, affs in [
    ("living_room", ["sofa", "floor"], ["somewhere to sit", "free space on the floor"]),
    ("road_sign", ["road", "truck"], ["free space to drive", "somewhere safe to stop"]),
    ("desk", ["desk", "keyboard"], ["somewhere to put a mug", "free space on the desk"]),
]:
    r = affordance_probe(seg, IMAGES[name], nouns, affs)
    for kind in ("nouns", "affordances"):
        for c, v in r[kind].items():
            print(f"  {name:<12} [{kind[:4]}] {c:<28} presence={v['presence']:<8} "
                  f"instances={v['n_instances']}")
    print()
    log.add("sam3", name, "affordance", prompt=str(affs),
            noun_presence=list(r["nouns"].values())[0]["presence"],
            affordance_presence=list(r["affordances"].values())[0]["presence"])

Look at *how* the affordance queries fail. A **low** presence score is not the
model saying "I don't understand the question" — it is the model asserting the
concept is absent. A room full of chairs contains nowhere to sit.

That is the sharpest limitation in this whole notebook for anyone specifying a
robot: the interface accepts any English string, so nothing warns you that
purpose-shaped requests silently return "not present". Any system that appears to
answer them is doing that reasoning **somewhere else in the stack**.

### Checkpoint

Before §4, write down two lists from your own numbers: failures §1 and §3 share,
and failures unique to one. The shared list is your evidence about the text
encoder. Remember both models' text encoders are CLIP-family — OWL-ViT's is CLIP,
this one's is MobileCLIP.

---
# §4 — SAM: the same masks, with the language taken out

SAM segments whatever is inside a **box**. It reads no text at all. Chain it to
OWL-ViT and you get the *detect-then-segment* pipeline most deployed robot stacks
still use, and a fair comparison against §3:

* **SAM 3** — text goes straight to instances.
* **OWL-ViT → SAM** — text only ever reaches the **box**; SAM refines geometry.

Both end in instance masks, so one number compares the pipelines: how much do
their masks agree for the same phrase?

In [ ]:
det = OwlDetector(device=DEVICE)             # re-created: §3 freed it
sam = SamSegmenter(device=DEVICE)            # ~375 MB, reads no text
print("loaded:", sam.model_id)
vram_report("SAM 3 + OWL-ViT + SAM resident")

In [ ]:
for name, phrase in [("cat", "a cat"), ("living_room", "a sofa"),
                     ("desk", "a keyboard"), ("fruit_bowl", "a lime")]:
    r = sam3_vs_owlvit_sam(det, seg, sam, IMAGES[name], phrase)
    print(f"{name:<12} {phrase!r:<12} agreement IoU={str(r['mask_agreement_iou']):<7} "
          f"sam3: {r['sam3_n_instances']} inst @ {r['sam3_top_score']} "
          f"(presence {r['sam3_presence']})  |  owlvit score={r['detector_score']}")
    log.add("sam3_vs_sam", name, "pipeline_agreement", prompt=phrase,
            mask_agreement_iou=r["mask_agreement_iou"],
            sam3_n_instances=r["sam3_n_instances"],
            detector_score=r["detector_score"])

In [ ]:
name, phrase = "desk", "a keyboard"
r = sam3_vs_owlvit_sam(det, seg, sam, IMAGES[name], phrase)
panels = {"SAM 3 (text -> instance)": r["sam3_mask"]}
if r["sam_mask"] is not None:
    panels["SAM (OWL-ViT box -> pixels)"] = r["sam_mask"]
show_mask_grid(IMAGES[name], panels,
               suptitle=f"{name} · {phrase!r} · agreement IoU = {r['mask_agreement_iou']}",
               ncols=2)

You will probably find the agreement **high** — the two pipelines draw nearly the
same pixels. That is the useful result, and it is worth being clear about why it
is useful, because it is easy to misread as "these two are interchangeable".

What the high agreement tells you is that **mask geometry is not where these
pipelines differ**. Both are good at turning a correct box into correct pixels.
So if you are choosing between them, boundary quality is not the deciding
factor — and any evaluation that only reports mask IoU would find them
equivalent and tell you nothing.

The difference is in the **language path**, and it does not show up in this
number at all:

* In §3, text reaches the model directly, and the model can report a
  **presence** score — it has a way to say "that concept is not here".
* Here, text only ever reaches the **box**. There is no presence output; SAM
  segments whatever rectangle it is handed. Every naming failure you measured in
  §1 is inherited silently, and SAM will faithfully trace a beautiful outline
  around it.

**A perfect boundary around the wrong object is still the wrong object** — and
this pipeline has no channel through which it could ever tell you so. Re-run the
comparison with a phrase for something *absent* and watch what each side does
with it. That, not the IoU, is the reason to care where the language sits.

In [ ]:
# ==== EDIT ME — free-form segmentation sandbox ===============================
SEG_TARGET     = "fruit_bowl"
SEG_CONCEPTS   = ["lime", "lemon", "bowl", "somewhere to grasp"]
SEG_THRESHOLD  = 0.5
# =============================================================================

img = IMAGES[SEG_TARGET]
for c in SEG_CONCEPTS:
    insts = seg.segment(img, c, threshold=SEG_THRESHOLD)
    box, owl_score = det.top_box(img, c, threshold=0.05)
    print(f"  {c:<24} presence={seg.presence(img, c):<8} "
          f"instances={len(insts):<3} top={insts[0].score if insts else 0.0:<7} "
          f"owlvit_score={owl_score}")
show_instance_comparison(img, [(c, seg.segment(img, c, threshold=SEG_THRESHOLD))
                               for c in SEG_CONCEPTS[:3]],
                         suptitle=f"SAM 3 · {SEG_TARGET}")

In [ ]:
# SAM has done its job -- it is only used in this section. Hand back its VRAM
# before the VLM section, which is by far the largest allocation in the notebook.
free_model(sam)
vram_report("after freeing SAM")

### The VRAM budget, out loud

A free T4 has roughly **15 GB**. This notebook drives four models, and the way it
fits is not luck — each section releases what it no longer needs, and the
`[vram]` lines above are there so you can check that rather than trust it.

If you ever hit an out-of-memory error, the cause is almost always re-running a
load cell twice without freeing in between. `free_model(...)` then re-running the
load cell fixes it; **Runtime > Restart** always fixes it.

---
# §5 — The VLM: text in, text out

A VLM can be *asked* things the other two heads cannot express: counts,
relations, comparisons. That flexibility is also new attack surface, because now
the failure can be a fluent, confident sentence.

### Four backends — and the choice affects your VRAM budget

| | Backend | VRAM | Needs |
|---|---|---|---|
| **A** | **OpenRouter** — `google/gemma-4-26b-a4b-it:free` | **0 GB** | a free API key |
| **B** | `SmolVLM-Instruct` (local) | **~4.9 GB** | a GPU runtime |
| **C** | Gemini (hosted, free tier) | 0 GB | a Google API key |

Every option is a **real model**. An offline mock used to sit here; it was
removed because its answers were a hash of the question text, so any pattern it
produced was an artifact of that rule rather than evidence about a VLM — and a
number that cannot support a claim does not belong in a lab whose deliverable is
claims.

The **hosted** paths run the whole section over HTTP, which leaves the entire T4
to the vision models and is the difference between a peak of roughly 8 GB and
roughly 3 GB. If you are sharing a GPU or hit an out-of-memory error, take
option A.

The hosted paths also make "does my finding survive a **different** VLM?" a
one-line change, and that is a much stronger claim than anything measured on a
single model. If you have time for only one extension, make it that one.

**If you did Week 9**, you already have an `OPENROUTER_API_KEY` Colab secret and
option A needs no setup — the key is resolved the same way, in the same order.

In [ ]:
free_model(det, sam, seg)
vram_report("all vision models freed", reset_peak=True)

# ============================ PICK ONE BACKEND ===============================

# --- A. OpenRouter, hosted free tier (0 GB VRAM) -----------------------------
# Key resolution: Colab Secrets -> environment -> hidden prompt (same as Week 9).
# PIN THIS: set a model that ACCEPTS IMAGES. Most free models are text-only and
# will fail or, worse, answer without having seen the picture.
#   list_free_vision_models()      # <- run this to see what is free right now
# OPENROUTER_VLM_MODEL = "google/gemma-4-26b-a4b-it:free"   # PINNED, verified live
# vlm = OpenRouterVLM(model_id=OPENROUTER_VLM_MODEL)

# --- B. Local SmolVLM (~4.9 GB, needs a GPU runtime) -------------------------
vlm = HFVLM(device=DEVICE)

# --- C. Hosted Gemini free tier (0 GB VRAM) ----------------------------------
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-generativeai"])
# from google.colab import userdata
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# vlm = GeminiVLM(model_id="")     # PIN THIS to a current free-tier Flash id

print("backend:", vlm.model_id)
vram_report("after VLM backend chosen")

> **Sanity-check a hosted backend before you trust it.** Ask it something about
> the image that a text-only model could not possibly answer, and confirm the
> answer is actually right:
>
> ```python
> print(vlm.ask(IMAGES["road_sign"], "What does the white sign say?"))
> ```
>
> A model that never received the image will still reply fluently. If that
> happens and you do not notice, every number in this section is measuring a
> blindfolded model — and it will look like a spectacular hallucination result.

### 5.1 The v1 probe: four phrasings of one yes/no question

This is the probe the v1 lab was built around. Run it first — including on
objects that are **not** in the image.

In [ ]:
img = IMAGES["fruit_bowl"]
r = phrasing_flip_probe(vlm, img, ["lime", "apple", "lemon", "giraffe"])
print(f"flip rate = {r['flip_rate']}  ({r['n_flipped']}/{r['n_objs']} objects "
      f"changed answer with wording alone)")
show_vlm_trials(r["trials"], limit=8)
log.add("vlm", "fruit_bowl", "phrasing_flip", prompt="4 templates x 4 objects",
        flip_rate=r["flip_rate"])

If your flip rate is **0.0**, that is a real result, not a failed experiment —
report it. It also means the v1 probe has run out of power on natural images,
and you need questions with more structure. The next three have that structure,
and each one is **self-checking**: you do not need to know what is in the photo
to catch the model contradicting itself.

### 5.2 Negation pairs — the model cannot have it both ways

"Is there a X?" and "Is there no X?" must get opposite answers. Same answer to
both is a contradiction, whatever is actually in the image.

In [ ]:
img = IMAGES["cat"]
r = negation_pair_probe(vlm, img, ["cat", "dog", "window"])
print(f"contradiction rate = {r['contradiction_rate']} "
      f"({r['n_contradictions']}/{r['n_scored']} scored pairs)")
show_vlm_trials(r["trials"], limit=6)
log.add("vlm", "cat", "negation_pair", prompt="cat/dog/window",
        contradiction_rate=r["contradiction_rate"])

### 5.3 Spatial-relation pairs — mutually exclusive by construction

"Is A to the left of B?" and "Is B to the left of A?" cannot both be *yes*.
Two yeses is an impossible pair of answers about any image.

Week 5's system-card audit had a capability report whose `spatial_relation`
success rate was **0 %**. This is that failure, measured directly.

In [ ]:
for name, pairs in [("fruit_bowl", [("lime", "orange"), ("apple", "banana")]),
                    ("desk", [("keyboard", "laptop"), ("hand", "keyboard")])]:
    r = spatial_pair_probe(vlm, IMAGES[name], pairs)
    print(f"\n{name}: incoherence rate = {r['incoherence_rate']} "
          f"(both-yes {r['n_both_yes']}, both-no {r['n_both_no']}, of {r['n_scored']})")
    show_vlm_trials(r["trials"], limit=4)
    log.add("vlm", name, "spatial_pair", prompt=str(pairs),
            incoherence_rate=r["incoherence_rate"], n_both_yes=r["n_both_yes"])

### 5.4 Presupposition traps — stop offering an escape hatch

A yes/no question lets a model hedge. A question that **presupposes** the object
("How many X are there?", "What colour is the X?") does not: answering at all
concedes that X exists.

`capitulation_rate` counts answers with no refusal marker. It is a crude keyword
rule — **read the raw text yourself** and say in your report where the rule
misjudged an answer.

In [ ]:
print("--- counting template ---")
for name, absent in [("fruit_bowl", ["lemons", "grapefruits"]), ("desk", ["giraffes"])]:
    r = presupposition_probe(vlm, IMAGES[name], absent)
    print(f"{name}: capitulation rate = {r['capitulation_rate']}")
    for t in r["trials"]:
        print(f"    [{t.parsed:<12}] {t.question:<42} -> {t.raw[:58]!r}")

print("\n--- colour template (presupposes harder) ---")
for name, absent in [("fruit_bowl", ["lemon"]), ("cat", ["dog"]), ("desk", ["giraffe"])]:
    r = presupposition_probe(vlm, IMAGES[name], absent,
                             template="What colour is the {obj} in the image?")
    for t in r["trials"]:
        print(f"    [{t.parsed:<12}] {name:<12} {t.question:<44} -> {t.raw[:52]!r}")
    log.add("vlm", name, "presupposition_colour", prompt=str(absent),
            capitulation_rate=r["capitulation_rate"])

Two things to pull out of your own output:

1. **Which template elicited more?** If the colour template capitulates where
   the counting template refused, then "does this model hallucinate" was never a
   property of the model alone — it was a property of the *question you asked*.
2. **Which absent objects got played along with?** Compare a plausible-in-scene
   absentee against an absurd one. If the model refuses giraffes on a desk but
   invents a lemon in a fruit bowl, then the driver is not absence — it is
   **contextual plausibility**. That is a much more dangerous failure, because
   the hallucinations you get are exactly the ones that look reasonable.

### Checkpoint — name the checkpoint, then try to break your own result

Every backend here is a real model, so your numbers mean something — but they
mean something about **one model**. Before writing anything down:

1. **Name the checkpoint** behind each number. On the hosted backend run
   `vlm.served_report()`: providers can route your request elsewhere, and a
   result you cannot attribute is not falsifiable.
2. **Re-run §5 on a second backend.** It is a one-line change, and it is the
   difference between "VLMs do this" and "this VLM does this" — which support
   opposite engineering decisions. If a failure survives two different models,
   say so; if it vanishes, say that too, because it means the failure was a
   property of a checkpoint.

---
# §6 — The showdown: one absent object, three heads

Everything so far has been within one model. Now ask the **same question about
the same absent object** to all three, and see whether the failures are
independent.

If the heads fail independently, disagreement is useful — you could cross-check
one against another. If they fail *together*, then stacking them buys you
nothing, because they inherited the same prior from the same backbone.

The three heads do not have equal equipment for this question, which is the
point: only SAM 3 has a head whose job is to say "absent". Ask whether having
that head is enough.

In [ ]:
img = IMAGES["fruit_bowl"]     # a lime, an orange, an apple, a banana. No lemon.
PRESENT, ABSENT = "a lime", "a lemon"

det = OwlDetector(device=DEVICE)     # §5 freed these to make room for the VLM
seg = Sam3Segmenter(device=DEVICE)
# This is the notebook's high-water mark: with a LOCAL VLM all three heads are
# resident at once. On a hosted backend the VLM contributes 0 GB here.
vram_report("PEAK — all heads resident")

print("== OWL-ViT (boxes) — no abstain output at all ==")
print("   present 'a lime' :", det.max_scores(img, [PRESENT])[PRESENT])
print("   ABSENT  'a lemon':", det.max_scores(img, [ABSENT])[ABSENT])

print("== SAM 3 (instances + a presence head) ==")
for p in (PRESENT, ABSENT):
    insts = seg.segment(img, p, threshold=0.5)
    low = seg.segment(img, p, threshold=0.05)
    print(f"   {p:<12} presence={seg.presence(img, p):<8} "
          f"instances@0.5={len(insts):<3} instances@0.05={len(low)}")

print("== SmolVLM (words) ==")
for q in ["Is there a lemon in the image? Answer yes or no.",
          "How many lemons are in the image?",
          "What colour is the lemon in the image?"]:
    print(f"   {q:<48} -> {vlm.ask(img, q)[:56]!r}")

show_instance_comparison(img, [
    (f"{PRESENT!r} @0.5", seg.segment(img, PRESENT, threshold=0.5)),
    (f"{ABSENT!r} @0.5", seg.segment(img, ABSENT, threshold=0.5)),
    (f"{ABSENT!r} @0.05", seg.segment(img, ABSENT, threshold=0.05)),
], suptitle="A fruit that is there, and one that is not")

### What to make of a split verdict

You will probably not find three identical failures here, and the shape of the
disagreement is more interesting than agreement would have been. Sort your own
numbers into these three cases:

1. **Fails outright** — reports the absent object as present.
2. **Has the candidates but suppresses them** — nothing at the default cut,
   something once you lower it. The model's *threshold* abstained.
3. **Genuinely absent** — nothing at any cut, and a low presence score.

The distinction between 2 and 3 is the one worth arguing about, because a system
card cannot tell them apart: both print "no lemon detected". Only one of them
would still say that after someone tuned a threshold for higher recall.

And note which object all three struggle with. It is not a random absent
object — a lemon is exactly what a model with a strong scene prior would expect
in a fruit bowl. Compare against how each head handled an *absurd* absentee
(`a car`, `a giraffe`). If the plausible one is harder for all three, that is
evidence about the **prior**, not about any one head.

### Fill this in from your own run — it is the core of the deliverable

| Failure | OWL-ViT (box) | SAM 3 (instance) | SmolVLM (word) | Head or backbone? |
|---|---|---|---|---|
| Cannot abstain on an absent object | | | | |
| Negation ignored | | | | |
| Attribute binding | | | | |
| Superordinate categories ("an animal") | | | | |
| Spatial relations | | | | |
| Output depends on a threshold you chose | | | | |
| Output depends on the *other* prompts you sent | | | | |

**How to use the last column.** A failure that appears in all three heads, in
the same direction, on the same object, is evidence for the **shared
backbone** — and it means adding another CLIP-derived model to your stack will
not catch it. A failure only one head shows is a **head** problem, and is
plausibly fixable by changing that head, its threshold, or its post-processing.

You have one piece of mechanistic evidence available, and you should use it
rather than argue from agreement alone: **all three text encoders are
CLIP-family** — CLIP in OWL-ViT, MobileCLIP in this SAM 3 variant, and a
CLIP-derived vision-language encoder in SmolVLM. So a failure that lives in the
*text* side (negation, superordinates, purpose-shaped phrases) has a named
common cause, while a failure in the *output* side (thresholds, instance
merging, prompt competition) does not.

Be careful with the inference, though. Three models agreeing is *consistent*
with a shared prior; it does not prove one. Say what else could explain it —
different training data with similar biases would look much the same — and what
you would have to run to tell those explanations apart.

---
# §7 — Your deliverable: a max-5-page PDF report

**Submit one PDF, 5 pages maximum.** The cap is hard and includes every table,
chart, and screenshot. **Condense:** one figure that *compares* conditions beats
three screenshots and a wall of numbers. Report only numbers your own run
produced, and name the model checkpoint behind each one.

In [ ]:
# Your logged experiments, as a markdown table for the report.
print(log.to_markdown())
# Or keep the raw records:  print(log.to_json())

## Worksheet (your deliverable)

### 1. The consistency table

One row per probe you ran, filled in with **your** numbers. These are the
label-free metrics — none of them needed an annotation.

| Model | Probe | Metric | Your value | What it means here |
|---|---|---|---|---|
| OWL-ViT | template sensitivity | `score_spread` | | |
| OWL-ViT | synonyms | `top_box_iou`, `score_delta` | | |
| OWL-ViT | negation | `iou_affirm_vs_negated` | | |
| OWL-ViT | absent objects | `separation_margin` | | |
| OWL-ViT | vocabulary competition | `boxes_lost_by_target` | | |
| SAM 3 | presence separation | `separation_margin` | | |
| SAM 3 | abstention vs threshold | `n_instances` at 0.05 vs 0.5 | | |
| SAM 3 | phrasing | `mean_pairwise_iou`, `count_spread` | | |
| SAM 3 | negation | `presence_cost_of_not` | | |
| SAM 3 | superordinates | `presence` for basic vs superordinate | | |
| SAM 3 | part/whole | `containment`, `area_ratio` | | |
| SAM 3 | head disagreement | `presence` with 0 instances | | |
| SAM 3 | affordances | `presence` for noun vs purpose phrasing | | |
| SAM 3 vs OWL-ViT+SAM | pipeline agreement | `mask_agreement_iou` | | |
| SmolVLM | phrasing | `flip_rate` | | |
| SmolVLM | negation pairs | `contradiction_rate` | | |
| SmolVLM | spatial pairs | `incoherence_rate` | | |
| SmolVLM | presupposition | `capitulation_rate` | | |

### 2. Edge-case portfolio (the graded core) — **4 cases minimum**

Pick your **four most interesting** edge cases. At least **two must come from
images you supplied yourself**. For each one:

1. **The input** — image (or a crop) and the exact prompt/question text.
2. **The output** — the box/mask/answer, rendered, with the score or metric.
3. **The minimal contrast** — the smallest change that fixes or breaks it. One
   word swapped, one threshold moved, one competing prompt added. *A case
   without a contrast is a screenshot, not a finding.*
4. **Classification** — head-specific or shared-backbone, **and your evidence**.
5. **The check that would have caught it** — a concrete evaluation: what data it
   runs on, and what threshold it must clear. "Test more" scores zero, exactly
   as in the Week 5 rubric.

### 3. The measurement lesson

- Which of your findings would a **fixed-class benchmark** (COCO-style mAP)
  have completely missed? Name one and say why the metric structurally cannot
  see it.
- Pick a number you reported and state what you would have to fix to make it
  **comparable** to the same number from another model.
- Did any probe find **nothing**? Say so, and say whether that means the model
  is sound or your inputs were too easy. (v1's lesson, and it still applies: a
  failure mode you care about may not surface unless the evaluation inputs are
  hard enough.)

### 4. The robot question

You are specifying perception for a mobile manipulator that takes
natural-language instructions. Using **only your own measurements**, give:

- one instruction phrasing you would **ban** from the interface, and the number
  that justifies banning it;
- one thing you would refuse to let the open-vocabulary stack decide on its own,
  and what you would put in front of it instead.

## How to improve this assignment (required, ungraded)

*Required for a complete submission; it carries no marks.* In 3–5 sentences:
what was unclear, too easy, too hard, or missing? Name the **one change** that
would make this a better learning exercise or a fairer test of the skill — a
different probe, a harder image, a model swap, a metric that would have caught
something this one missed, or a clearer instruction. Be specific; "it was fine"
is not useful feedback.

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking

---
### Image credits

The six starter photos are CC0 or Public Domain Mark; run the image-bank cell to
print full provenance (title, creator, license, source URL). Reproduce that
attribution if you republish a figure built on them. Images you upload yourself
are your own responsibility — do not submit photos of identifiable people
without their consent.

### Going further

- **OWLv2** (`google/owlv2-base-patch16-ensemble`) is a drop-in swap for
  `OwlDetector(model_id=...)` — the stronger detector the lecture also cites.
  Re-run §1 and see which findings survive a better model. The ones that do are
  the interesting ones.
- **`sam-vit-large`** for cleaner boundaries, if VRAM allows.
- Swap the VLM checkpoint and re-run §5. A failure that persists across two
  different VLMs is a much stronger claim than one measured on a single model.
- **CLIPSeg** (`CIDAS/clipseg-rd64-refined`) is the *semantic* counterpart to §3's
  instance model — text straight to a per-pixel heatmap, no instances, no
  presence head, and only ~150 MB. Worth running if you want to see what the
  presence head buys you: on the cat photo it paints roughly the same area for
  `"a dog"` as for `"a cat"`. It is also a 2022 model, so it doubles as a
  before/after on three years of progress.
- **`facebook/sam3`** is the official Meta checkpoint this variant is distilled
  from. It is **gated**: accept the licence on its model page, then set an
  `HF_TOKEN` Colab secret. Worth doing if you want a result you can attribute to
  an official release rather than a third-party distillation.